# EcoSignal — complete standalone backend (Features 1–16)

This is the cumulative, self-contained Google Colab backend. It includes every tested feature from the stable sample schema through the unified analysis pipeline and model-independent confidence aggregation. It preserves the central scientific rule: **predictions are immutable, corrections remain separate, and only reviewed annotations enter explicit dataset versions**.

The notebook uses Google Drive for the persistent SQLite database, image/mask assets, exports, and an unpacked copy of the backend code. No ZIP or additional source file is required.


## 1. Install Colab dependencies

In [ ]:
%pip install -q "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "sqlalchemy>=2.0,<3" "pydantic>=2.8,<3" "httpx>=0.27,<1" "Pillow>=10.4,<12" "python-multipart>=0.0.9,<1" "pytest>=8.3,<9"

## 2. Mount Google Drive and configure persistent storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ECOSIGNAL_ROOT = Path('/content/drive/MyDrive/EcoSignal')
DATA_DIR = ECOSIGNAL_ROOT / 'data'
ASSET_ROOT = ECOSIGNAL_ROOT / 'assets'
PACKAGE_ROOT = ECOSIGNAL_ROOT / 'backend_features_1_16'
for directory in (DATA_DIR, ASSET_ROOT, PACKAGE_ROOT): directory.mkdir(parents=True, exist_ok=True)
DATABASE_PATH = DATA_DIR / 'ecosignal.db'
DATABASE_URL = f'sqlite:///{DATABASE_PATH}'
print('Database:', DATABASE_PATH)
print('Scientific assets and exports:', ASSET_ROOT)
print('Backend code:', PACKAGE_ROOT)


## 3. Materialize the complete embedded backend

This automatically adds the new Feature 6–16 tables and the two new reviewer fields to an existing Features 1–5 Drive database. Existing samples, predictions, assets, corrections, reviews, and staging items are preserved.


In [ ]:
import sys
PACKAGE_FILES = {'app/__init__.py': '"""EcoSignal standalone backend package."""\n\nfrom .main import create_app\n\n__all__ = ["create_app"]\n\n', 'app/database.py': '"""Database construction and request-scoped sessions."""\n\nfrom __future__ import annotations\n\nimport os\nfrom collections.abc import Generator\n\nfrom sqlalchemy import create_engine\nfrom sqlalchemy.engine import Engine\nfrom sqlalchemy.orm import DeclarativeBase, Session, sessionmaker\n\n\nDEFAULT_DATABASE_URL = "sqlite:///./ecosignal.db"\n\n\nclass Base(DeclarativeBase):\n    """Base class shared by all SQLAlchemy models."""\n\n\ndef build_engine(database_url: str | None = None) -> Engine:\n    """Build an engine that works with both SQLite and later databases."""\n\n    url = database_url or os.getenv("ECOSIGNAL_DATABASE_URL", DEFAULT_DATABASE_URL)\n    connect_args = {"check_same_thread": False} if url.startswith("sqlite") else {}\n    return create_engine(url, connect_args=connect_args, future=True)\n\n\ndef build_session_factory(engine: Engine) -> sessionmaker[Session]:\n    return sessionmaker(bind=engine, autoflush=False, expire_on_commit=False)\n\n\ndef session_dependency(\n    session_factory: sessionmaker[Session],\n) -> Generator[Session, None, None]:\n    session = session_factory()\n    try:\n        yield session\n    finally:\n        session.close()\n\n', 'app/models.py': '"""Cumulative EcoSignal database models through Feature 16.\n\nModel-specific results deliberately live in a JSON column. This prevents the\ncoverage, growth-form, chlorosis, and future adapters from creating mutually\nincompatible prediction tables.\n"""\n\nfrom __future__ import annotations\n\nfrom datetime import datetime\nfrom typing import Any\nfrom uuid import uuid4\n\nfrom sqlalchemy import Boolean, DateTime, Float, ForeignKey, Index, Integer, JSON, String, Text, UniqueConstraint\nfrom sqlalchemy.orm import Mapped, mapped_column, relationship\n\nfrom .database import Base\n\n\ndef new_uuid() -> str:\n    return str(uuid4())\n\n\ndef utc_now() -> datetime:\n    return datetime.now().astimezone()\n\n\nclass Sample(Base):\n    __tablename__ = "samples"\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    study_area_id: Mapped[str | None] = mapped_column(String(36), nullable=True, index=True)\n    user_id: Mapped[str | None] = mapped_column(String(128), nullable=True, index=True)\n    captured_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False)\n    latitude: Mapped[float | None] = mapped_column(Float, nullable=True)\n    longitude: Mapped[float | None] = mapped_column(Float, nullable=True)\n    substrate_type: Mapped[str | None] = mapped_column(String(64), nullable=True)\n    site_type: Mapped[str | None] = mapped_column(String(64), nullable=True)\n    notes: Mapped[str | None] = mapped_column(Text, nullable=True)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="submitted")\n    created_at: Mapped[datetime] = mapped_column(\n        DateTime(timezone=True), nullable=False, default=utc_now\n    )\n    updated_at: Mapped[datetime] = mapped_column(\n        DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now\n    )\n\n    predictions: Mapped[list["Prediction"]] = relationship(\n        back_populates="sample", cascade="all, delete-orphan"\n    )\n    assets: Mapped[list["SampleAsset"]] = relationship(\n        back_populates="sample", cascade="all, delete-orphan"\n    )\n    corrections: Mapped[list["Correction"]] = relationship(\n        back_populates="sample", cascade="all, delete-orphan"\n    )\n\n\nclass Prediction(Base):\n    __tablename__ = "predictions"\n    __table_args__ = (\n        Index("ix_predictions_sample_model", "sample_id", "model_type"),\n    )\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(\n        ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    model_type: Mapped[str] = mapped_column(String(64), nullable=False)\n    model_version: Mapped[str] = mapped_column(String(128), nullable=False)\n    predicted_value_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False)\n    confidence: Mapped[float | None] = mapped_column(Float, nullable=True)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="completed")\n    warnings_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    is_placeholder: Mapped[bool] = mapped_column(nullable=False, default=False)\n    created_at: Mapped[datetime] = mapped_column(\n        DateTime(timezone=True), nullable=False, default=utc_now\n    )\n\n    sample: Mapped[Sample] = relationship(back_populates="predictions")\n    corrections: Mapped[list["Correction"]] = relationship(\n        back_populates="prediction", cascade="all, delete-orphan"\n    )\n\n\nclass SampleAsset(Base):\n    """Immutable metadata for an original or derived scientific image."""\n\n    __tablename__ = "sample_assets"\n    __table_args__ = (\n        UniqueConstraint("sample_id", "checksum", name="uq_asset_sample_checksum"),\n        Index("ix_assets_sample_created", "sample_id", "created_at"),\n    )\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(\n        ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    asset_type: Mapped[str] = mapped_column(String(64), nullable=False)\n    storage_path: Mapped[str] = mapped_column(Text, nullable=False)\n    mime_type: Mapped[str] = mapped_column(String(128), nullable=False)\n    checksum: Mapped[str] = mapped_column(String(128), nullable=False, index=True)\n    width: Mapped[int | None] = mapped_column(nullable=True)\n    height: Mapped[int | None] = mapped_column(nullable=True)\n    created_by: Mapped[str | None] = mapped_column(String(128), nullable=True)\n    source_asset_id: Mapped[str | None] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    original_filename: Mapped[str | None] = mapped_column(String(255), nullable=True)\n    created_at: Mapped[datetime] = mapped_column(\n        DateTime(timezone=True), nullable=False, default=utc_now\n    )\n\n    sample: Mapped[Sample] = relationship(back_populates="assets")\n\n\nclass Correction(Base):\n    """A user proposal stored separately from the immutable model prediction."""\n\n    __tablename__ = "corrections"\n    __table_args__ = (\n        Index("ix_corrections_status_created", "status", "created_at"),\n        Index("ix_corrections_prediction_created", "prediction_id", "created_at"),\n    )\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(\n        ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    prediction_id: Mapped[str] = mapped_column(\n        ForeignKey("predictions.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    field_name: Mapped[str] = mapped_column(String(128), nullable=False)\n    original_value_json: Mapped[Any] = mapped_column(JSON, nullable=True)\n    proposed_value_json: Mapped[Any] = mapped_column(JSON, nullable=True)\n    proposed_asset_id: Mapped[str | None] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    correction_method: Mapped[str | None] = mapped_column(String(64), nullable=True)\n    reason: Mapped[str] = mapped_column(Text, nullable=False)\n    submitted_by: Mapped[str] = mapped_column(String(128), nullable=False)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="pending_review")\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n    updated_at: Mapped[datetime] = mapped_column(\n        DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now\n    )\n    withdrawn_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True), nullable=True)\n\n    sample: Mapped[Sample] = relationship(back_populates="corrections")\n    prediction: Mapped[Prediction] = relationship(back_populates="corrections")\n    reviews: Mapped[list["Review"]] = relationship(\n        back_populates="correction", cascade="all, delete-orphan"\n    )\n    staging_item: Mapped["DatasetStagingItem | None"] = relationship(\n        back_populates="correction", cascade="all, delete-orphan", uselist=False\n    )\n\n\nclass Review(Base):\n    """Immutable reviewer decision; correction status is the derived workflow state."""\n\n    __tablename__ = "reviews"\n    __table_args__ = (\n        UniqueConstraint("correction_id", "reviewer_id", name="uq_review_correction_reviewer"),\n        Index("ix_reviews_correction_reviewed", "correction_id", "reviewed_at"),\n    )\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    correction_id: Mapped[str] = mapped_column(\n        ForeignKey("corrections.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    reviewer_id: Mapped[str] = mapped_column(String(128), nullable=False, index=True)\n    decision: Mapped[str] = mapped_column(String(32), nullable=False)\n    reviewer_notes: Mapped[str | None] = mapped_column(Text, nullable=True)\n    reviewed_value_json: Mapped[Any] = mapped_column(JSON, nullable=True)\n    assigned_dataset_version_id: Mapped[str | None] = mapped_column(String(36), nullable=True)\n    reviewed_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n    correction: Mapped[Correction] = relationship(back_populates="reviews")\n\n\nclass DatasetStagingItem(Base):\n    """Approved annotation awaiting explicit dataset versioning in Feature 6."""\n\n    __tablename__ = "dataset_staging_items"\n    __table_args__ = (Index("ix_staging_sample_created", "sample_id", "staged_at"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    correction_id: Mapped[str] = mapped_column(\n        ForeignKey("corrections.id", ondelete="RESTRICT"), nullable=False, unique=True, index=True\n    )\n    sample_id: Mapped[str] = mapped_column(\n        ForeignKey("samples.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    prediction_id: Mapped[str] = mapped_column(\n        ForeignKey("predictions.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    field_name: Mapped[str] = mapped_column(String(128), nullable=False)\n    verified_value_json: Mapped[Any] = mapped_column(JSON, nullable=True)\n    annotation_asset_id: Mapped[str | None] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    annotation_source: Mapped[str] = mapped_column(String(64), nullable=False, default="approved_user_correction")\n    approved_by_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="ready")\n    staged_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n    correction: Mapped[Correction] = relationship(back_populates="staging_item")\n\n\nclass DatasetVersion(Base):\n    """A reproducible, explicitly versioned collection of reviewed annotations."""\n\n    __tablename__ = "dataset_versions"\n    __table_args__ = (UniqueConstraint("name", name="uq_dataset_version_name"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    name: Mapped[str] = mapped_column(String(128), nullable=False, index=True)\n    parent_version_id: Mapped[str | None] = mapped_column(\n        ForeignKey("dataset_versions.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="draft")\n    description: Mapped[str | None] = mapped_column(Text, nullable=True)\n    created_by: Mapped[str] = mapped_column(String(128), nullable=False)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n    frozen_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True), nullable=True)\n    manifest_checksum: Mapped[str | None] = mapped_column(String(64), nullable=True)\n\n\nclass DatasetItem(Base):\n    __tablename__ = "dataset_items"\n    __table_args__ = (\n        UniqueConstraint("dataset_version_id", "sample_id", name="uq_dataset_version_sample"),\n        Index("ix_dataset_items_version_split", "dataset_version_id", "split"),\n    )\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    dataset_version_id: Mapped[str] = mapped_column(\n        ForeignKey("dataset_versions.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    staging_item_id: Mapped[str] = mapped_column(\n        ForeignKey("dataset_staging_items.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    sample_id: Mapped[str] = mapped_column(ForeignKey("samples.id", ondelete="RESTRICT"), nullable=False, index=True)\n    image_asset_id: Mapped[str] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    annotation_asset_id: Mapped[str | None] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    split: Mapped[str] = mapped_column(String(16), nullable=False)\n    annotation_source: Mapped[str] = mapped_column(String(64), nullable=False)\n    approved_correction_id: Mapped[str] = mapped_column(\n        ForeignKey("corrections.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    site_group: Mapped[str] = mapped_column(String(128), nullable=False, index=True)\n    verified_value_json: Mapped[Any] = mapped_column(JSON, nullable=True)\n    added_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n\nclass DatasetExport(Base):\n    __tablename__ = "dataset_exports"\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    dataset_version_id: Mapped[str] = mapped_column(\n        ForeignKey("dataset_versions.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    export_format: Mapped[str] = mapped_column(String(32), nullable=False)\n    storage_path: Mapped[str] = mapped_column(Text, nullable=False)\n    checksum: Mapped[str] = mapped_column(String(64), nullable=False)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="completed")\n    item_count: Mapped[int] = mapped_column(Integer, nullable=False)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n\nclass AuditEvent(Base):\n    """Append-only provenance event. No update or delete endpoint is exposed."""\n\n    __tablename__ = "audit_events"\n    __table_args__ = (Index("ix_audit_events_entity_created", "entity_type", "entity_id", "created_at"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    event_type: Mapped[str] = mapped_column(String(64), nullable=False, index=True)\n    entity_type: Mapped[str] = mapped_column(String(64), nullable=False)\n    entity_id: Mapped[str] = mapped_column(String(36), nullable=False, index=True)\n    sample_id: Mapped[str | None] = mapped_column(String(36), nullable=True, index=True)\n    actor_id: Mapped[str | None] = mapped_column(String(128), nullable=True)\n    payload_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False, default=dict)\n    previous_event_hash: Mapped[str | None] = mapped_column(String(64), nullable=True)\n    event_hash: Mapped[str] = mapped_column(String(64), nullable=False, unique=True)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n\nclass AnalysisRun(Base):\n    __tablename__ = "analysis_runs"\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(ForeignKey("samples.id", ondelete="RESTRICT"), nullable=False, index=True)\n    input_asset_id: Mapped[str] = mapped_column(\n        ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=False, index=True\n    )\n    input_asset_checksum: Mapped[str] = mapped_column(String(64), nullable=False)\n    pipeline_version: Mapped[str] = mapped_column(String(128), nullable=False)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="running")\n    requested_models_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    model_versions_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False, default=dict)\n    geospatial_layer_versions_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False, default=dict)\n    warnings_json: Mapped[list[dict[str, Any]]] = mapped_column(JSON, nullable=False, default=list)\n    results_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False, default=dict)\n    started_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n    completed_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True), nullable=True)\n\n\nclass AnalysisStep(Base):\n    __tablename__ = "analysis_steps"\n    __table_args__ = (Index("ix_analysis_steps_run_order", "analysis_run_id", "sequence"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    analysis_run_id: Mapped[str] = mapped_column(\n        ForeignKey("analysis_runs.id", ondelete="CASCADE"), nullable=False, index=True\n    )\n    sequence: Mapped[int] = mapped_column(Integer, nullable=False)\n    step_name: Mapped[str] = mapped_column(String(64), nullable=False)\n    status: Mapped[str] = mapped_column(String(32), nullable=False)\n    version: Mapped[str | None] = mapped_column(String(128), nullable=True)\n    output_json: Mapped[dict[str, Any] | None] = mapped_column(JSON, nullable=True)\n    warning_json: Mapped[dict[str, Any] | None] = mapped_column(JSON, nullable=True)\n    error_json: Mapped[dict[str, Any] | None] = mapped_column(JSON, nullable=True)\n    started_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n    completed_at: Mapped[datetime | None] = mapped_column(DateTime(timezone=True), nullable=True)\n\n\nclass ModelRegistryEntry(Base):\n    __tablename__ = "model_registry"\n    __table_args__ = (UniqueConstraint("model_type", "version", name="uq_model_type_version"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    model_type: Mapped[str] = mapped_column(String(64), nullable=False, index=True)\n    version: Mapped[str] = mapped_column(String(128), nullable=False, index=True)\n    status: Mapped[str] = mapped_column(String(32), nullable=False)\n    dataset_version_id: Mapped[str | None] = mapped_column(\n        ForeignKey("dataset_versions.id", ondelete="RESTRICT"), nullable=True, index=True\n    )\n    artifact_path: Mapped[str | None] = mapped_column(Text, nullable=True)\n    validation_status: Mapped[str] = mapped_column(String(32), nullable=False)\n    card_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False)\n    metrics_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False, default=dict)\n    threshold: Mapped[float | None] = mapped_column(Float, nullable=True)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n    updated_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now)\n\n\nclass ActiveLearningItem(Base):\n    __tablename__ = "active_learning_items"\n    __table_args__ = (UniqueConstraint("sample_id", "queue_type", name="uq_active_learning_sample_queue"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True)\n    queue_type: Mapped[str] = mapped_column(String(32), nullable=False, index=True)\n    priority_score: Mapped[float] = mapped_column(Float, nullable=False, index=True)\n    priority_reasons_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    components_json: Mapped[dict[str, float]] = mapped_column(JSON, nullable=False, default=dict)\n    formula_version: Mapped[str] = mapped_column(String(64), nullable=False)\n    status: Mapped[str] = mapped_column(String(32), nullable=False, default="queued")\n    updated_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now, onupdate=utc_now)\n\n\nclass QualityAssessment(Base):\n    __tablename__ = "quality_assessments"\n    __table_args__ = (Index("ix_quality_asset_created", "asset_id", "created_at"),)\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True)\n    asset_id: Mapped[str] = mapped_column(ForeignKey("sample_assets.id", ondelete="RESTRICT"), nullable=False, index=True)\n    quality_score: Mapped[float] = mapped_column(Float, nullable=False)\n    decision: Mapped[str] = mapped_column(String(32), nullable=False)\n    metrics_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False)\n    warnings_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    thresholds_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False)\n    quality_version: Mapped[str] = mapped_column(String(64), nullable=False)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n\n\nclass ConfidenceAssessment(Base):\n    __tablename__ = "confidence_assessments"\n\n    id: Mapped[str] = mapped_column(String(36), primary_key=True, default=new_uuid)\n    sample_id: Mapped[str] = mapped_column(ForeignKey("samples.id", ondelete="CASCADE"), nullable=False, index=True)\n    analysis_run_id: Mapped[str | None] = mapped_column(\n        ForeignKey("analysis_runs.id", ondelete="SET NULL"), nullable=True, index=True\n    )\n    components_json: Mapped[dict[str, Any]] = mapped_column(JSON, nullable=False)\n    overall_confidence: Mapped[float | None] = mapped_column(Float, nullable=True)\n    weights_json: Mapped[dict[str, float]] = mapped_column(JSON, nullable=False)\n    limitations_json: Mapped[list[str]] = mapped_column(JSON, nullable=False, default=list)\n    confidence_version: Mapped[str] = mapped_column(String(64), nullable=False)\n    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), nullable=False, default=utc_now)\n', 'app/schemas.py': '"""Pydantic request and response schemas through Feature 16."""\n\nfrom __future__ import annotations\n\nfrom datetime import datetime\nfrom typing import Any, Annotated\n\nfrom pydantic import BaseModel, ConfigDict, Field, StringConstraints, model_validator\n\n\nNonEmptyString = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1)]\n\n\nclass ApiModel(BaseModel):\n    model_config = ConfigDict(from_attributes=True, extra="forbid")\n\n\nclass SampleCreate(ApiModel):\n    study_area_id: str | None = None\n    user_id: str | None = None\n    captured_at: datetime\n    latitude: float | None = Field(default=None, ge=-90, le=90)\n    longitude: float | None = Field(default=None, ge=-180, le=180)\n    substrate_type: str | None = Field(default=None, max_length=64)\n    site_type: str | None = Field(default=None, max_length=64)\n    notes: str | None = Field(default=None, max_length=4000)\n    status: NonEmptyString = "submitted"\n\n    @model_validator(mode="after")\n    def coordinates_are_paired(self) -> "SampleCreate":\n        if (self.latitude is None) != (self.longitude is None):\n            raise ValueError("latitude and longitude must be supplied together")\n        return self\n\n\nclass SampleUpdate(ApiModel):\n    study_area_id: str | None = None\n    user_id: str | None = None\n    captured_at: datetime | None = None\n    latitude: float | None = Field(default=None, ge=-90, le=90)\n    longitude: float | None = Field(default=None, ge=-180, le=180)\n    substrate_type: str | None = Field(default=None, max_length=64)\n    site_type: str | None = Field(default=None, max_length=64)\n    notes: str | None = Field(default=None, max_length=4000)\n    status: NonEmptyString | None = None\n\n    @model_validator(mode="after")\n    def coordinates_are_paired_when_both_in_payload(self) -> "SampleUpdate":\n        fields = self.model_fields_set\n        if ("latitude" in fields) != ("longitude" in fields):\n            raise ValueError("patch latitude and longitude together")\n        if "latitude" in fields and ((self.latitude is None) != (self.longitude is None)):\n            raise ValueError("latitude and longitude must both be values or both be null")\n        return self\n\n\nclass SampleRead(ApiModel):\n    sample_id: str = Field(validation_alias="id")\n    study_area_id: str | None\n    user_id: str | None\n    captured_at: datetime\n    latitude: float | None\n    longitude: float | None\n    substrate_type: str | None\n    site_type: str | None\n    notes: str | None\n    status: str\n    created_at: datetime\n    updated_at: datetime\n\n\nclass PredictionCreate(ApiModel):\n    model_type: NonEmptyString\n    model_version: NonEmptyString\n    predicted_value: dict[str, Any]\n    confidence: float | None = Field(default=None, ge=0, le=1)\n    status: NonEmptyString = "completed"\n    warnings: list[str] = Field(default_factory=list)\n    is_placeholder: bool = False\n\n    @model_validator(mode="after")\n    def placeholder_is_explicit(self) -> "PredictionCreate":\n        if "placeholder" in self.model_version.lower() and not self.is_placeholder:\n            raise ValueError(\n                "placeholder model versions must set is_placeholder=true"\n            )\n        return self\n\n\nclass PredictionRead(ApiModel):\n    prediction_id: str = Field(validation_alias="id")\n    sample_id: str\n    model_type: str\n    model_version: str\n    predicted_value: dict[str, Any] = Field(validation_alias="predicted_value_json")\n    confidence: float | None\n    status: str\n    warnings: list[str] = Field(validation_alias="warnings_json")\n    is_placeholder: bool\n    created_at: datetime\n\n\nclass AssetRead(ApiModel):\n    asset_id: str = Field(validation_alias="id")\n    sample_id: str\n    asset_type: str\n    storage_path: str\n    mime_type: str\n    checksum: str\n    width: int | None\n    height: int | None\n    created_by: str | None\n    source_asset_id: str | None\n    original_filename: str | None\n    created_at: datetime\n\n\nclass CoverageAnalysisRequest(ApiModel):\n    image_asset_id: NonEmptyString\n    requested_by: NonEmptyString = "api-user"\n\n\nclass CorrectionCreate(ApiModel):\n    field_name: NonEmptyString\n    proposed_value: Any | None = None\n    proposed_asset_id: str | None = None\n    correction_method: str | None = Field(default=None, max_length=64)\n    reason: str = Field(min_length=10, max_length=4000)\n    submitted_by: NonEmptyString\n\n\nclass CorrectionRead(ApiModel):\n    correction_id: str = Field(validation_alias="id")\n    sample_id: str\n    prediction_id: str\n    field_name: str\n    original_value: Any | None = Field(validation_alias="original_value_json")\n    proposed_value: Any | None = Field(validation_alias="proposed_value_json")\n    proposed_asset_id: str | None\n    correction_method: str | None\n    reason: str\n    submitted_by: str\n    status: str\n    created_at: datetime\n    updated_at: datetime\n    withdrawn_at: datetime | None\n\n\nclass CorrectionWithdrawRequest(ApiModel):\n    requested_by: NonEmptyString\n\n\nclass ReviewDecisionCreate(ApiModel):\n    reviewer_id: NonEmptyString\n    decision: str\n    reviewer_notes: str | None = Field(default=None, max_length=4000)\n    adjusted_value: Any | None = None\n    dataset_version_id: str | None = None\n\n    @model_validator(mode="after")\n    def valid_decision(self) -> "ReviewDecisionCreate":\n        allowed = {"approved", "rejected", "needs_more_evidence"}\n        self.decision = self.decision.strip().lower()\n        if self.decision not in allowed:\n            raise ValueError(f"decision must be one of: {\', \'.join(sorted(allowed))}")\n        if self.decision != "approved" and not (self.reviewer_notes or "").strip():\n            raise ValueError("rejected and needs_more_evidence decisions require reviewer_notes")\n        return self\n\n\nclass ReviewRead(ApiModel):\n    review_id: str = Field(validation_alias="id")\n    correction_id: str\n    reviewer_id: str\n    decision: str\n    reviewer_notes: str | None\n    reviewed_value: Any | None = Field(validation_alias="reviewed_value_json")\n    assigned_dataset_version_id: str | None\n    reviewed_at: datetime\n\n\nclass DatasetStagingRead(ApiModel):\n    staging_item_id: str = Field(validation_alias="id")\n    correction_id: str\n    sample_id: str\n    prediction_id: str\n    field_name: str\n    verified_value: Any | None = Field(validation_alias="verified_value_json")\n    annotation_asset_id: str | None\n    annotation_source: str\n    approved_by: list[str] = Field(validation_alias="approved_by_json")\n    status: str\n    staged_at: datetime\n\n\nclass ReviewQueueItem(ApiModel):\n    correction_id: str\n    sample_id: str\n    prediction_id: str\n    field_name: str\n    original_value: Any | None\n    proposed_value: Any | None\n    proposed_asset_id: str | None\n    reason: str\n    correction_status: str\n    prediction_confidence: float | None\n    approvals_received: int\n    approvals_required: int\n    created_at: datetime\n\n\nclass ReviewDetail(ApiModel):\n    correction: CorrectionRead\n    prediction: PredictionRead\n    sample: SampleRead\n    assets: list[AssetRead]\n    previous_corrections: list[CorrectionRead]\n    reviews: list[ReviewRead]\n    staging_item: DatasetStagingRead | None\n    review_requirements: dict[str, Any]\n\n\nclass ReviewDecisionResult(ApiModel):\n    correction: CorrectionRead\n    review: ReviewRead\n    staging_item: DatasetStagingRead | None\n    approvals_received: int\n    approvals_required: int\n\n\nclass DatasetVersionCreate(ApiModel):\n    name: str = Field(min_length=3, max_length=128, pattern=r"^[A-Za-z0-9_.-]+$")\n    parent_version_id: str | None = None\n    description: str | None = Field(default=None, max_length=4000)\n    created_by: NonEmptyString\n\n\nclass DatasetVersionRead(ApiModel):\n    dataset_version_id: str = Field(validation_alias="id")\n    name: str\n    parent_version_id: str | None\n    status: str\n    description: str | None\n    created_by: str\n    created_at: datetime\n    frozen_at: datetime | None\n    manifest_checksum: str | None\n\n\nclass DatasetItemCreate(ApiModel):\n    staging_item_id: NonEmptyString\n    image_asset_id: NonEmptyString\n    split: str\n    site_group: str | None = Field(default=None, max_length=128)\n\n    @model_validator(mode="after")\n    def valid_split(self) -> "DatasetItemCreate":\n        self.split = self.split.strip().lower()\n        if self.split not in {"train", "validation", "test"}:\n            raise ValueError("split must be train, validation, or test")\n        return self\n\n\nclass DatasetItemRead(ApiModel):\n    dataset_item_id: str = Field(validation_alias="id")\n    dataset_version_id: str\n    staging_item_id: str\n    sample_id: str\n    image_asset_id: str\n    annotation_asset_id: str | None\n    split: str\n    annotation_source: str\n    approved_correction_id: str\n    site_group: str\n    verified_value: Any | None = Field(validation_alias="verified_value_json")\n    added_at: datetime\n\n\nclass DatasetExportCreate(ApiModel):\n    export_format: str = "zip"\n\n    @model_validator(mode="after")\n    def valid_format(self) -> "DatasetExportCreate":\n        self.export_format = self.export_format.strip().lower()\n        if self.export_format not in {"zip", "csv", "json", "geojson"}:\n            raise ValueError("export_format must be zip, csv, json, or geojson")\n        return self\n\n\nclass DatasetExportRead(ApiModel):\n    export_id: str = Field(validation_alias="id")\n    dataset_version_id: str\n    export_format: str\n    storage_path: str\n    checksum: str\n    status: str\n    item_count: int\n    created_at: datetime\n\n\nclass AuditEventRead(ApiModel):\n    event_id: str = Field(validation_alias="id")\n    event_type: str\n    entity_type: str\n    entity_id: str\n    sample_id: str | None\n    actor_id: str | None\n    payload: dict[str, Any] = Field(validation_alias="payload_json")\n    previous_event_hash: str | None\n    event_hash: str\n    created_at: datetime\n\n\nclass ModelRegistryCreate(ApiModel):\n    model_type: NonEmptyString\n    version: NonEmptyString\n    status: str\n    dataset_version_id: str | None = None\n    artifact_path: str | None = Field(default=None, max_length=1000)\n    validation_status: NonEmptyString = "pending"\n    card: dict[str, Any]\n    metrics: dict[str, Any] = Field(default_factory=dict)\n    threshold: float | None = Field(default=None, ge=0, le=1)\n\n    @model_validator(mode="after")\n    def validate_registry(self) -> "ModelRegistryCreate":\n        allowed = {"development", "validation_candidate", "production_candidate", "production", "retired"}\n        self.status = self.status.strip().lower()\n        if self.status not in allowed:\n            raise ValueError(f"status must be one of: {\', \'.join(sorted(allowed))}")\n        required = {"name", "purpose", "input", "output", "training_data", "validation_data", "known_failure_cases", "intended_use", "prohibited_use"}\n        missing = sorted(required - self.card.keys())\n        if missing:\n            raise ValueError(f"model card missing fields: {\', \'.join(missing)}")\n        return self\n\n\nclass ModelRegistryUpdate(ApiModel):\n    status: str | None = None\n    dataset_version_id: str | None = None\n    artifact_path: str | None = Field(default=None, max_length=1000)\n    validation_status: str | None = None\n    card: dict[str, Any] | None = None\n    metrics: dict[str, Any] | None = None\n    threshold: float | None = Field(default=None, ge=0, le=1)\n\n\nclass ModelRegistryRead(ApiModel):\n    model_id: str = Field(validation_alias="id")\n    model_type: str\n    version: str\n    status: str\n    dataset_version_id: str | None\n    artifact_path: str | None\n    validation_status: str\n    card: dict[str, Any] = Field(validation_alias="card_json")\n    metrics: dict[str, Any] = Field(validation_alias="metrics_json")\n    threshold: float | None\n    created_at: datetime\n    updated_at: datetime\n\n\nclass ActiveLearningRecalculate(ApiModel):\n    formula_version: NonEmptyString = "active_learning_priority_v0"\n    weights: dict[str, float] = Field(default_factory=lambda: {\n        "model_uncertainty": .30, "geographic_uncertainty": .25,\n        "class_rarity": .20, "sample_quality": .15, "study_importance": .10,\n    })\n\n    @model_validator(mode="after")\n    def valid_weights(self) -> "ActiveLearningRecalculate":\n        expected = {"model_uncertainty", "geographic_uncertainty", "class_rarity", "sample_quality", "study_importance"}\n        if set(self.weights) != expected or any(not 0 <= value <= 1 for value in self.weights.values()):\n            raise ValueError("weights must provide all five components with values from 0 to 1")\n        if abs(sum(self.weights.values()) - 1.0) > 1e-6:\n            raise ValueError("active-learning weights must sum to 1")\n        return self\n\n\nclass ActiveLearningUpdate(ApiModel):\n    status: str\n\n    @model_validator(mode="after")\n    def valid_status(self) -> "ActiveLearningUpdate":\n        self.status = self.status.strip().lower()\n        if self.status not in {"queued", "assigned", "completed", "dismissed"}:\n            raise ValueError("invalid active-learning status")\n        return self\n\n\nclass ActiveLearningRead(ApiModel):\n    item_id: str = Field(validation_alias="id")\n    sample_id: str\n    queue_type: str\n    priority_score: float\n    priority_reasons: list[str] = Field(validation_alias="priority_reasons_json")\n    components: dict[str, float] = Field(validation_alias="components_json")\n    formula_version: str\n    status: str\n    updated_at: datetime\n\n\nclass ReanalysisRequest(ApiModel):\n    asset_id: NonEmptyString\n    model_type: NonEmptyString\n    model_version: NonEmptyString\n\n\nclass QualityAssessmentRead(ApiModel):\n    quality_assessment_id: str = Field(validation_alias="id")\n    sample_id: str\n    asset_id: str\n    quality_score: float\n    decision: str\n    metrics: dict[str, Any] = Field(validation_alias="metrics_json")\n    warnings: list[str] = Field(validation_alias="warnings_json")\n    thresholds: dict[str, Any] = Field(validation_alias="thresholds_json")\n    quality_version: str\n    created_at: datetime\n\n\nclass AnalyzeSampleRequest(ApiModel):\n    sample_id: NonEmptyString\n    asset_id: NonEmptyString\n    run_models: list[str] = Field(min_length=1)\n    pipeline_version: NonEmptyString = "ecosignal_pipeline_v1"\n    geospatial_layer_versions: dict[str, str] = Field(default_factory=dict)\n\n\nclass ConfidenceCreate(ApiModel):\n    analysis_run_id: str | None = None\n    components: dict[str, float | None]\n    confidence_version: NonEmptyString = "sample_confidence_v0"\n    weights: dict[str, float] | None = None\n\n\nclass ConfidenceRead(ApiModel):\n    confidence_assessment_id: str = Field(validation_alias="id")\n    sample_id: str\n    analysis_run_id: str | None\n    components: dict[str, Any] = Field(validation_alias="components_json")\n    overall_confidence: float | None\n    weights: dict[str, float] = Field(validation_alias="weights_json")\n    limitations: list[str] = Field(validation_alias="limitations_json")\n    confidence_version: str\n    created_at: datetime\n', 'app/inference.py': '"""Coverage-model interface and explicitly non-scientific placeholder adapter."""\n\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom hashlib import sha256\nfrom io import BytesIO\nimport os\nfrom pathlib import Path\nfrom typing import Any\n\nfrom PIL import Image\n\n\nclass CoverageModel(ABC):\n    """Stable interface implemented by both placeholder and final models."""\n\n    model_version: str\n    is_placeholder: bool\n\n    @abstractmethod\n    def predict(self, image_path: str) -> dict[str, Any]:\n        """Return the shared coverage result structure for one image."""\n\n\nclass PlaceholderCoverageModel(CoverageModel):\n    """Deterministic development adapter; never represents scientific output."""\n\n    model_version = "coverage_placeholder_v0"\n    is_placeholder = True\n\n    def predict(self, image_path: str) -> dict[str, Any]:\n        path = Path(image_path)\n        image_bytes = path.read_bytes()\n        digest = sha256(image_bytes).digest()\n        coverage_percent = round(25.0 + (int.from_bytes(digest[:2], "big") / 65535) * 35.0, 1)\n\n        with Image.open(path) as source:\n            width, height = source.size\n\n        # A deterministic development-only mask keeps the end-to-end asset flow\n        # testable without claiming that pixels were scientifically segmented.\n        target_pixels = round(width * height * coverage_percent / 100)\n        mask = Image.new("L", (width, height), 0)\n        pixels = mask.load()\n        offset = int.from_bytes(digest[2:4], "big") % max(1, width * height)\n        stride = 104729  # prime: spreads selected pixels across common image sizes\n        selected: set[int] = set()\n        cursor = offset\n        while len(selected) < target_pixels:\n            selected.add(cursor % (width * height))\n            cursor += stride\n        for index in selected:\n            pixels[index % width, index // width] = 255\n\n        buffer = BytesIO()\n        mask.save(buffer, format="PNG")\n        return {\n            "coverage_percent": coverage_percent,\n            "confidence": 0.70,\n            "mask_bytes": buffer.getvalue(),\n            "mask_mime_type": "image/png",\n            "model_version": self.model_version,\n            "is_placeholder": True,\n            "warnings": [\n                "DEVELOPMENT_PLACEHOLDER: coverage and mask are not scientific results"\n            ],\n        }\n\n\ndef get_coverage_model() -> CoverageModel:\n    """Configuration switch used by the API\'s dependency-injection boundary."""\n\n    configured = os.getenv("ECOSIGNAL_COVERAGE_MODEL", "placeholder").strip().lower()\n    if configured == "placeholder":\n        return PlaceholderCoverageModel()\n    raise RuntimeError(\n        f"Coverage model \'{configured}\' is not registered. Pass a CoverageModel "\n        "implementation to create_app or select ECOSIGNAL_COVERAGE_MODEL=placeholder."\n    )\n', 'app/services.py': '"""Model-independent services shared by EcoSignal Features 8, 9, 14, and 16."""\n\nfrom __future__ import annotations\n\nfrom hashlib import sha256\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nfrom PIL import Image, ImageFilter, ImageStat\nfrom sqlalchemy import select\nfrom sqlalchemy.orm import Session\n\nfrom .models import AuditEvent, utc_now\n\n\nERROR_CATALOG: dict[str, dict[str, Any]] = {\n    "UNSUPPORTED_FILE_TYPE": {"severity": "error", "retry_allowed": True},\n    "IMAGE_CORRUPTED": {"severity": "error", "retry_allowed": True},\n    "IMAGE_TOO_SMALL": {"severity": "error", "retry_allowed": True},\n    "IMAGE_TOO_BLURRY": {"severity": "error", "retry_allowed": True},\n    "IMAGE_TOO_DARK": {"severity": "error", "retry_allowed": True},\n    "IMAGE_OVEREXPOSED": {"severity": "error", "retry_allowed": True},\n    "DUPLICATE_IMAGE": {"severity": "error", "retry_allowed": False},\n    "MODEL_NOT_AVAILABLE": {"severity": "error", "retry_allowed": False},\n    "MODEL_TIMEOUT": {"severity": "error", "retry_allowed": True},\n    "MODEL_INTERNAL_ERROR": {"severity": "error", "retry_allowed": True},\n    "NO_LICHEN_DETECTED": {"severity": "error", "retry_allowed": True},\n    "PREDICTION_LOW_CONFIDENCE": {"severity": "warning", "retry_allowed": False},\n    "GROWTH_FORM_REJECTED": {"severity": "warning", "retry_allowed": True},\n    "CHLOROSIS_UNAVAILABLE": {"severity": "warning", "retry_allowed": False},\n    "NITROPHYTE_ACIDOPHYTE_UNKNOWN": {"severity": "warning", "retry_allowed": False},\n    "FEATURE_SCHEMA_MISMATCH": {"severity": "error", "retry_allowed": False},\n    "LOCATION_MISSING": {"severity": "warning", "retry_allowed": True},\n    "LOCATION_UNSUPPORTED": {"severity": "error", "retry_allowed": False},\n    "GEOSPATIAL_LOOKUP_FAILED": {"severity": "warning", "retry_allowed": True},\n    "MAP_CELL_NOT_FOUND": {"severity": "warning", "retry_allowed": True},\n    "ASSET_UPLOAD_FAILED": {"severity": "error", "retry_allowed": True},\n    "SAMPLE_NOT_FOUND": {"severity": "error", "retry_allowed": False},\n    "DATABASE_WRITE_FAILED": {"severity": "error", "retry_allowed": True},\n    "LOW_LIGHT_WARNING": {"severity": "warning", "retry_allowed": True},\n    "MISSING_OPTIONAL_METADATA": {"severity": "warning", "retry_allowed": True},\n    "LOW_GROWTH_FORM_CONFIDENCE": {"severity": "warning", "retry_allowed": False},\n}\n\n\ndef issue(code: str, message: str, *, details: dict[str, Any] | None = None,\n          status: str | None = None) -> dict[str, Any]:\n    properties = ERROR_CATALOG.get(code, {"severity": "error", "retry_allowed": False})\n    return {\n        "status": status or ("warning" if properties["severity"] == "warning" else "failed"),\n        "code": code,\n        "message": message,\n        "severity": properties["severity"],\n        "retry_allowed": properties["retry_allowed"],\n        "details": details or {},\n    }\n\n\ndef append_event(session: Session, event_type: str, entity_type: str, entity_id: str,\n                 *, sample_id: str | None = None, actor_id: str | None = None,\n                 payload: dict[str, Any] | None = None) -> AuditEvent:\n    """Append a hash-chained event; callers commit it with the related transaction."""\n    previous = session.scalar(select(AuditEvent).order_by(AuditEvent.created_at.desc(), AuditEvent.id.desc()).limit(1))\n    created_at = utc_now()\n    canonical = json.dumps({\n        "event_type": event_type, "entity_type": entity_type, "entity_id": entity_id,\n        "sample_id": sample_id, "actor_id": actor_id, "payload": payload or {},\n        "previous_event_hash": previous.event_hash if previous else None,\n    }, sort_keys=True, separators=(",", ":"), default=str)\n    event = AuditEvent(\n        event_type=event_type, entity_type=entity_type, entity_id=entity_id,\n        sample_id=sample_id, actor_id=actor_id, payload_json=payload or {},\n        previous_event_hash=previous.event_hash if previous else None,\n        event_hash=sha256(canonical.encode("utf-8")).hexdigest(), created_at=created_at,\n    )\n    session.add(event)\n    session.flush()\n    return event\n\n\nclass ModelAdapter(Protocol):\n    model_type: str\n    model_version: str\n    is_placeholder: bool\n\n    def predict(self, image_path: str, context: dict[str, Any] | None = None) -> dict[str, Any]: ...\n\n\nclass QualityService:\n    quality_version = "image_quality_v0_configurable"\n\n    def __init__(self, thresholds: dict[str, float] | None = None):\n        defaults = {\n            "min_width": float(os.getenv("ECOSIGNAL_MIN_IMAGE_WIDTH", "64")),\n            "min_height": float(os.getenv("ECOSIGNAL_MIN_IMAGE_HEIGHT", "64")),\n            "min_blur_score": float(os.getenv("ECOSIGNAL_MIN_BLUR_SCORE", "8")),\n            "min_brightness": float(os.getenv("ECOSIGNAL_MIN_BRIGHTNESS", "0.10")),\n            "max_brightness": float(os.getenv("ECOSIGNAL_MAX_BRIGHTNESS", "0.92")),\n            "max_overexposed_fraction": float(os.getenv("ECOSIGNAL_MAX_OVEREXPOSED", "0.80")),\n            "min_contrast": float(os.getenv("ECOSIGNAL_MIN_CONTRAST", "0.08")),\n        }\n        defaults.update(thresholds or {})\n        self.thresholds = defaults\n\n    def assess(self, image_path: str | Path, metadata_complete: bool = True) -> dict[str, Any]:\n        with Image.open(image_path) as image:\n            gray = image.convert("L")\n            width, height = gray.size\n            stats = ImageStat.Stat(gray)\n            brightness = stats.mean[0] / 255.0\n            contrast = stats.stddev[0] / 255.0\n            edges = gray.filter(ImageFilter.FIND_EDGES)\n            blur_score = ImageStat.Stat(edges).var[0]\n            histogram = gray.histogram()\n            overexposed_fraction = sum(histogram[245:]) / max(1, width * height)\n\n        metrics = {\n            "width": width, "height": height, "blur_score": round(blur_score, 4),\n            "brightness": round(brightness, 4), "contrast": round(contrast, 4),\n            "overexposed_fraction": round(overexposed_fraction, 4),\n            "metadata_completeness": 1.0 if metadata_complete else 0.0,\n        }\n        blockers: list[str] = []\n        warnings: list[str] = []\n        t = self.thresholds\n        if width < t["min_width"] or height < t["min_height"]: blockers.append("IMAGE_TOO_SMALL")\n        if blur_score < t["min_blur_score"]: blockers.append("IMAGE_TOO_BLURRY")\n        if brightness < t["min_brightness"]: blockers.append("IMAGE_TOO_DARK")\n        if brightness > t["max_brightness"] or overexposed_fraction > t["max_overexposed_fraction"]: blockers.append("IMAGE_OVEREXPOSED")\n        if contrast < t["min_contrast"]: warnings.append("LOW_CONTRAST")\n        if not metadata_complete: warnings.append("MISSING_OPTIONAL_METADATA")\n\n        component_scores = [\n            min(1.0, width / max(1.0, t["min_width"])),\n            min(1.0, height / max(1.0, t["min_height"])),\n            min(1.0, blur_score / max(1.0, t["min_blur_score"] * 2)),\n            max(0.0, 1.0 - abs(brightness - .5) / .5),\n            min(1.0, contrast / max(.01, t["min_contrast"] * 2)),\n            1.0 if metadata_complete else .5,\n        ]\n        quality_score = round(sum(component_scores) / len(component_scores), 4)\n        decision = "rejected" if blockers else ("accepted_with_warning" if warnings else "accepted")\n        return {\n            "quality_score": quality_score, "decision": decision, "metrics": metrics,\n            "warnings": blockers + warnings, "thresholds": dict(t),\n            "quality_version": self.quality_version,\n        }\n\n\nDEFAULT_CONFIDENCE_WEIGHTS = {\n    "image_quality": .20, "coverage": .20, "growth_form": .15,\n    "chlorosis": .10, "geospatial_completeness": .15,\n    "risk_engine": .15, "feature_completeness": .05,\n}\n\n\ndef aggregate_confidence(components: dict[str, float | None],\n                         weights: dict[str, float] | None = None) -> dict[str, Any]:\n    selected = dict(weights or DEFAULT_CONFIDENCE_WEIGHTS)\n    unknown = set(components) - set(selected)\n    if unknown:\n        raise ValueError(f"Unknown confidence components: {\', \'.join(sorted(unknown))}")\n    if any(value is not None and not 0 <= value <= 1 for value in components.values()):\n        raise ValueError("Confidence components must be null or between 0 and 1")\n    present = {key: float(value) for key, value in components.items() if value is not None and key in selected}\n    denominator = sum(selected[key] for key in present)\n    overall = None if denominator == 0 else round(sum(present[key] * selected[key] for key in present) / denominator, 4)\n    limitations = [f"{key.replace(\'_\', \' \').title()} confidence unavailable" for key in selected if components.get(key) is None]\n    return {"overall_confidence": overall, "weights": selected, "limitations": limitations}\n', 'app/remaining.py': '"""API routes for cumulative EcoSignal Features 6 through 16."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nfrom datetime import datetime\nfrom hashlib import sha256\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nimport shutil\nfrom typing import Any\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nfrom fastapi import Depends, FastAPI, HTTPException, Query, status\nfrom fastapi.responses import FileResponse\nfrom PIL import Image\nfrom sqlalchemy import func, select\nfrom sqlalchemy.exc import IntegrityError\nfrom sqlalchemy.orm import Session\n\nfrom .database import session_dependency\nfrom .models import (\n    ActiveLearningItem, AnalysisRun, AnalysisStep, AuditEvent, ConfidenceAssessment,\n    Correction, DatasetExport, DatasetItem, DatasetStagingItem, DatasetVersion,\n    ModelRegistryEntry, Prediction, QualityAssessment, Review, Sample, SampleAsset,\n    new_uuid, utc_now,\n)\nfrom .schemas import (\n    ActiveLearningRead, ActiveLearningRecalculate, ActiveLearningUpdate,\n    AnalyzeSampleRequest, AuditEventRead, ConfidenceCreate, ConfidenceRead,\n    DatasetExportCreate, DatasetExportRead, DatasetItemCreate, DatasetItemRead,\n    DatasetVersionCreate, DatasetVersionRead, ModelRegistryCreate, ModelRegistryRead,\n    ModelRegistryUpdate, PredictionRead, QualityAssessmentRead, ReanalysisRequest,\n)\nfrom .services import QualityService, aggregate_confidence, append_event, issue\n\n\ndef register_remaining_routes(app: FastAPI, session_factory, asset_root: Path,\n                              model_adapters: dict[str, Any], quality_service: QualityService) -> None:\n    export_root = asset_root / "exports"\n\n    def get_session():\n        yield from session_dependency(session_factory)\n\n    def require(session: Session, model, object_id: str, label: str):\n        value = session.get(model, object_id)\n        if value is None:\n            raise HTTPException(404, detail=issue(f"{label.upper()}_NOT_FOUND", f"{label.title()} not found"))\n        return value\n\n    def require_draft(version_id: str, session: Session) -> DatasetVersion:\n        version = require(session, DatasetVersion, version_id, "dataset_version")\n        if version.status != "draft":\n            raise HTTPException(409, detail=issue("DATASET_VERSION_IMMUTABLE", "Only draft dataset versions can be modified"))\n        return version\n\n    def asset_path(asset: SampleAsset) -> Path:\n        path = (asset_root / asset.storage_path).resolve()\n        if asset_root not in path.parents or not path.is_file():\n            raise HTTPException(422, detail=issue("ASSET_MISSING", "A required asset file is missing", details={"asset_id": asset.id}))\n        return path\n\n    def site_group_for(sample: Sample, requested: str | None) -> str:\n        if requested and requested.strip(): return requested.strip()\n        if sample.study_area_id: return f"study:{sample.study_area_id}"\n        if sample.latitude is not None and sample.longitude is not None:\n            return f"geo:{sample.latitude:.3f},{sample.longitude:.3f}"\n        return f"sample:{sample.id}"\n\n    def registry_for(session: Session, model_type: str, version: str) -> ModelRegistryEntry:\n        entry = session.scalar(select(ModelRegistryEntry).where(\n            ModelRegistryEntry.model_type == model_type, ModelRegistryEntry.version == version\n        ))\n        if entry is None:\n            raise HTTPException(422, detail=issue("MODEL_NOT_REGISTERED", "Model version must be registered before use", details={"model_type": model_type, "version": version}))\n        if entry.status == "retired":\n            raise HTTPException(409, detail=issue("MODEL_RETIRED", "Retired models remain traceable but cannot run new analyses"))\n        return entry\n\n    def save_mask(sample_id: str, source: SampleAsset, data: bytes, version: str, session: Session) -> SampleAsset:\n        checksum = sha256(data).hexdigest()\n        existing = session.scalar(select(SampleAsset).where(SampleAsset.sample_id == sample_id, SampleAsset.checksum == checksum))\n        if existing: return existing\n        with Image.open(BytesIO(data)) as image:\n            width, height = image.size\n            image.verify()\n        asset_id = new_uuid()\n        relative = Path("samples") / sample_id / f"{asset_id}_coverage_mask.png"\n        destination = (asset_root / relative).resolve()\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        destination.write_bytes(data)\n        mask = SampleAsset(id=asset_id, sample_id=sample_id, asset_type="coverage_mask",\n            storage_path=relative.as_posix(), mime_type="image/png", checksum=checksum,\n            width=width, height=height, created_by=version, source_asset_id=source.id,\n            original_filename=f"{version}_mask.png")\n        session.add(mask)\n        return mask\n\n    def normalize_adapter_result(model_type: str, result: dict[str, Any], source: SampleAsset,\n                                 session: Session) -> tuple[dict[str, Any], float | None, str, bool, list[str]]:\n        version = str(result.get("model_version") or getattr(model_adapters.get(model_type), "model_version", "unknown"))\n        placeholder = bool(result.get("is_placeholder", getattr(model_adapters.get(model_type), "is_placeholder", False)))\n        confidence = result.get("confidence")\n        if confidence is not None and not 0 <= float(confidence) <= 1:\n            raise ValueError("adapter confidence must be from 0 to 1")\n        warnings = list(result.get("warnings", []))\n        if "predicted_value" in result:\n            predicted = dict(result["predicted_value"])\n        else:\n            excluded = {"confidence", "model_version", "is_placeholder", "warnings", "mask_bytes", "mask_mime_type"}\n            predicted = {key: value for key, value in result.items() if key not in excluded}\n        if model_type == "coverage" and isinstance(result.get("mask_bytes"), bytes):\n            mask = save_mask(source.sample_id, source, result["mask_bytes"], version, session)\n            predicted["mask_asset_id"] = mask.id\n            predicted["source_asset_id"] = source.id\n        predicted["is_placeholder"] = placeholder\n        return predicted, float(confidence) if confidence is not None else None, version, placeholder, warnings\n\n    def run_adapter(model_type: str, source: SampleAsset, session: Session,\n                    context: dict[str, Any] | None = None, requested_version: str | None = None) -> Prediction:\n        adapter = model_adapters.get(model_type)\n        if adapter is None:\n            raise HTTPException(503, detail=issue("MODEL_NOT_AVAILABLE", f"{model_type} adapter is not connected"))\n        adapter_version = str(getattr(adapter, "model_version", "unknown"))\n        if requested_version and requested_version != adapter_version:\n            raise HTTPException(503, detail=issue("MODEL_NOT_AVAILABLE", "Requested model adapter is not connected", details={"requested_version": requested_version, "connected_version": adapter_version}))\n        registry_for(session, model_type, adapter_version)\n        try:\n            try: result = adapter.predict(str(asset_path(source)), context=context)\n            except TypeError: result = adapter.predict(str(asset_path(source)))\n            predicted, confidence, version, placeholder, warnings = normalize_adapter_result(model_type, result, source, session)\n        except HTTPException:\n            raise\n        except Exception as exc:\n            raise HTTPException(503, detail=issue("MODEL_INTERNAL_ERROR", f"{model_type} adapter failed", details={"exception_type": type(exc).__name__})) from exc\n        prediction = Prediction(sample_id=source.sample_id, model_type=model_type, model_version=version,\n            predicted_value_json=predicted, confidence=confidence, status="completed",\n            warnings_json=warnings, is_placeholder=placeholder)\n        session.add(prediction); session.flush()\n        append_event(session, "PREDICTION_CREATED", "prediction", prediction.id,\n            sample_id=source.sample_id, payload={"model_type": model_type, "model_version": version, "source_asset_id": source.id})\n        return prediction\n\n    def dataset_manifest(version: DatasetVersion, session: Session) -> dict[str, Any]:\n        rows = list(session.scalars(select(DatasetItem).where(DatasetItem.dataset_version_id == version.id).order_by(DatasetItem.split, DatasetItem.sample_id)))\n        items = []\n        for row in rows:\n            image = require(session, SampleAsset, row.image_asset_id, "asset")\n            annotation = session.get(SampleAsset, row.annotation_asset_id) if row.annotation_asset_id else None\n            items.append({"dataset_item_id": row.id, "sample_id": row.sample_id, "split": row.split,\n                "site_group": row.site_group, "image_asset_id": row.image_asset_id,\n                "image_checksum": image.checksum, "annotation_asset_id": row.annotation_asset_id,\n                "annotation_checksum": annotation.checksum if annotation else None,\n                "annotation_source": row.annotation_source, "approved_correction_id": row.approved_correction_id,\n                "verified_value": row.verified_value_json})\n        counts = Counter(item["split"] for item in items)\n        return {"dataset_version_id": version.id, "name": version.name, "parent_version_id": version.parent_version_id,\n            "status": version.status, "created_at": version.created_at.isoformat(), "frozen_at": version.frozen_at.isoformat() if version.frozen_at else None,\n            "counts": {"train": counts["train"], "validation": counts["validation"], "test": counts["test"]}, "items": items}\n\n    # Features 6 and 7: dataset versions, split safeguards, manifests, and exports.\n    @app.post("/dataset-versions", response_model=DatasetVersionRead, status_code=201)\n    def create_dataset_version(payload: DatasetVersionCreate, session: Session = Depends(get_session)):\n        if payload.parent_version_id: require(session, DatasetVersion, payload.parent_version_id, "dataset_version")\n        version = DatasetVersion(name=payload.name, parent_version_id=payload.parent_version_id,\n            description=payload.description, created_by=payload.created_by.strip(), status="draft")\n        session.add(version)\n        try:\n            session.flush(); append_event(session, "DATASET_VERSION_CREATED", "dataset_version", version.id, actor_id=version.created_by, payload={"name": version.name}); session.commit()\n        except IntegrityError as exc:\n            session.rollback(); raise HTTPException(409, detail=issue("DATASET_VERSION_EXISTS", "Dataset version name already exists")) from exc\n        session.refresh(version); return version\n\n    @app.get("/dataset-versions", response_model=list[DatasetVersionRead])\n    def list_dataset_versions(session: Session = Depends(get_session)):\n        return list(session.scalars(select(DatasetVersion).order_by(DatasetVersion.created_at, DatasetVersion.id)))\n\n    @app.post("/dataset-versions/{version_id}/items", response_model=DatasetItemRead, status_code=201)\n    def add_dataset_item(version_id: str, payload: DatasetItemCreate, session: Session = Depends(get_session)):\n        version = require_draft(version_id, session)\n        staging = require(session, DatasetStagingItem, payload.staging_item_id, "staging_item")\n        correction = require(session, Correction, staging.correction_id, "correction")\n        if staging.status not in {"ready", "versioned"} or correction.status != "approved":\n            raise HTTPException(422, detail=issue("ANNOTATION_NOT_APPROVED", "Only approved staged annotations can enter a dataset"))\n        sample = require(session, Sample, staging.sample_id, "sample")\n        image = require(session, SampleAsset, payload.image_asset_id, "asset")\n        if image.sample_id != sample.id or image.asset_type != "original_image":\n            raise HTTPException(422, detail=issue("INVALID_DATASET_IMAGE", "Dataset image must be the sample\'s original image"))\n        if staging.annotation_asset_id:\n            annotation = require(session, SampleAsset, staging.annotation_asset_id, "asset")\n            if annotation.sample_id != sample.id: raise HTTPException(422, detail=issue("INVALID_ANNOTATION", "Annotation belongs to another sample"))\n            if annotation.width != image.width or annotation.height != image.height:\n                raise HTTPException(422, detail=issue("ANNOTATION_DIMENSION_MISMATCH", "Mask and image dimensions must match"))\n        prior_test = session.scalar(select(DatasetItem).where(DatasetItem.sample_id == sample.id, DatasetItem.split == "test"))\n        if prior_test and payload.split != "test":\n            raise HTTPException(409, detail=issue("TEST_SET_IMMUTABLE", "A sample assigned to test can never move to training or validation"))\n        site_group = site_group_for(sample, payload.site_group)\n        leakage = session.scalar(select(DatasetItem).where(DatasetItem.dataset_version_id == version.id,\n            DatasetItem.site_group == site_group, DatasetItem.split != payload.split))\n        if leakage:\n            raise HTTPException(409, detail=issue("GEOGRAPHIC_SPLIT_LEAKAGE", "Samples in the same site group must remain in one split", details={"site_group": site_group, "existing_split": leakage.split}))\n        row = DatasetItem(dataset_version_id=version.id, staging_item_id=staging.id, sample_id=sample.id,\n            image_asset_id=image.id, annotation_asset_id=staging.annotation_asset_id, split=payload.split,\n            annotation_source=staging.annotation_source, approved_correction_id=correction.id,\n            site_group=site_group, verified_value_json=staging.verified_value_json)\n        session.add(row)\n        try:\n            session.flush(); staging.status = "versioned"; append_event(session, "DATASET_ITEM_ADDED", "dataset_item", row.id, sample_id=sample.id,\n                payload={"dataset_version_id": version.id, "split": row.split, "correction_id": correction.id}); session.commit()\n        except IntegrityError as exc:\n            session.rollback(); raise HTTPException(409, detail=issue("DUPLICATE_DATASET_SAMPLE", "This sample is already in the dataset version")) from exc\n        session.refresh(row); return row\n\n    @app.post("/dataset-versions/{version_id}/freeze", response_model=DatasetVersionRead)\n    def freeze_dataset(version_id: str, session: Session = Depends(get_session)):\n        version = require_draft(version_id, session)\n        manifest = dataset_manifest(version, session)\n        if not manifest["items"]: raise HTTPException(422, detail=issue("EMPTY_DATASET", "An empty dataset version cannot be frozen"))\n        version.status = "frozen"; version.frozen_at = utc_now()\n        manifest["status"] = "frozen"; manifest["frozen_at"] = version.frozen_at.isoformat()\n        version.manifest_checksum = sha256(json.dumps(manifest, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()\n        append_event(session, "DATASET_VERSION_FROZEN", "dataset_version", version.id, actor_id=version.created_by,\n            payload={"manifest_checksum": version.manifest_checksum, "counts": manifest["counts"]})\n        session.commit(); session.refresh(version); return version\n\n    @app.get("/dataset-versions/{version_id}/manifest")\n    def get_dataset_manifest(version_id: str, session: Session = Depends(get_session)):\n        return dataset_manifest(require(session, DatasetVersion, version_id, "dataset_version"), session)\n\n    @app.post("/dataset-versions/{version_id}/export", response_model=DatasetExportRead, status_code=201)\n    def export_dataset(version_id: str, payload: DatasetExportCreate, session: Session = Depends(get_session)):\n        version = require(session, DatasetVersion, version_id, "dataset_version")\n        if version.status != "frozen": raise HTTPException(409, detail=issue("DATASET_NOT_FROZEN", "Freeze the dataset before export"))\n        manifest = dataset_manifest(version, session); rows = list(session.scalars(select(DatasetItem).where(DatasetItem.dataset_version_id == version.id)))\n        destination_dir = export_root / f"{version.name}_{new_uuid()}"; destination_dir.mkdir(parents=True, exist_ok=False)\n        metadata_rows, features = [], []\n        for row in rows:\n            sample = require(session, Sample, row.sample_id, "sample"); image = require(session, SampleAsset, row.image_asset_id, "asset")\n            image_source = asset_path(image); image_rel = Path("images") / f"{row.sample_id}{image_source.suffix.lower()}"\n            (destination_dir / image_rel).parent.mkdir(parents=True, exist_ok=True); shutil.copy2(image_source, destination_dir / image_rel)\n            mask_rel = None\n            if row.annotation_asset_id:\n                annotation = require(session, SampleAsset, row.annotation_asset_id, "asset"); annotation_source = asset_path(annotation)\n                if annotation.width != image.width or annotation.height != image.height:\n                    shutil.rmtree(destination_dir); raise HTTPException(422, detail=issue("ANNOTATION_DIMENSION_MISMATCH", "Mask and image dimensions differ"))\n                mask_rel = Path("masks") / f"{row.sample_id}{annotation_source.suffix.lower()}"\n                (destination_dir / mask_rel).parent.mkdir(parents=True, exist_ok=True); shutil.copy2(annotation_source, destination_dir / mask_rel)\n            metadata_rows.append({"sample_id": row.sample_id, "image_path": image_rel.as_posix(), "mask_path": mask_rel.as_posix() if mask_rel else "",\n                "split": row.split, "source": row.annotation_source, "site_id": row.site_group, "verified_value": json.dumps(row.verified_value_json)})\n            features.append({"type": "Feature", "geometry": ({"type": "Point", "coordinates": [sample.longitude, sample.latitude]} if sample.latitude is not None else None),\n                "properties": {"sample_id": row.sample_id, "split": row.split, "site_group": row.site_group}})\n        (destination_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")\n        (destination_dir / "metadata.csv").parent.mkdir(exist_ok=True)\n        with (destination_dir / "metadata.csv").open("w", newline="", encoding="utf-8") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(metadata_rows[0])); writer.writeheader(); writer.writerows(metadata_rows)\n        (destination_dir / "records.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": features}, indent=2), encoding="utf-8")\n        if payload.export_format == "zip":\n            output = destination_dir.with_suffix(".zip")\n            with ZipFile(output, "w", ZIP_DEFLATED) as archive:\n                for file in sorted(destination_dir.rglob("*")):\n                    if file.is_file(): archive.write(file, file.relative_to(destination_dir))\n            shutil.rmtree(destination_dir)\n        else:\n            chosen = {"csv": "metadata.csv", "json": "manifest.json", "geojson": "records.geojson"}[payload.export_format]\n            output = destination_dir / chosen\n        checksum = sha256(output.read_bytes()).hexdigest()\n        relative = output.relative_to(asset_root).as_posix()\n        record = DatasetExport(dataset_version_id=version.id, export_format=payload.export_format,\n            storage_path=relative, checksum=checksum, status="completed", item_count=len(rows))\n        session.add(record); session.flush(); append_event(session, "DATASET_EXPORTED", "dataset_export", record.id,\n            payload={"dataset_version_id": version.id, "format": payload.export_format, "checksum": checksum}); session.commit(); session.refresh(record); return record\n\n    @app.get("/exports/{export_id}", response_model=DatasetExportRead)\n    def get_export(export_id: str, session: Session = Depends(get_session)):\n        return require(session, DatasetExport, export_id, "export")\n\n    @app.get("/exports/{export_id}/content", response_class=FileResponse)\n    def get_export_content(export_id: str, session: Session = Depends(get_session)):\n        record = require(session, DatasetExport, export_id, "export"); path = (asset_root / record.storage_path).resolve()\n        if asset_root not in path.parents or not path.is_file(): raise HTTPException(404, detail=issue("EXPORT_MISSING", "Export file not found"))\n        return FileResponse(path, filename=path.name)\n\n    # Feature 8: audit and analysis-run provenance.\n    @app.get("/audit-events", response_model=list[AuditEventRead])\n    def audit_events(sample_id: str | None = None, entity_id: str | None = None, session: Session = Depends(get_session)):\n        query = select(AuditEvent)\n        if sample_id: query = query.where(AuditEvent.sample_id == sample_id)\n        if entity_id: query = query.where(AuditEvent.entity_id == entity_id)\n        return list(session.scalars(query.order_by(AuditEvent.created_at, AuditEvent.id)))\n\n    @app.get("/audit-events/verify")\n    def verify_audit_chain(session: Session = Depends(get_session)):\n        events = list(session.scalars(select(AuditEvent).order_by(AuditEvent.created_at, AuditEvent.id)))\n        previous = None\n        for event in events:\n            canonical = json.dumps({\n                "event_type": event.event_type, "entity_type": event.entity_type,\n                "entity_id": event.entity_id, "sample_id": event.sample_id,\n                "actor_id": event.actor_id, "payload": event.payload_json or {},\n                "previous_event_hash": event.previous_event_hash,\n            }, sort_keys=True, separators=(",", ":"), default=str)\n            expected_hash = sha256(canonical.encode("utf-8")).hexdigest()\n            if event.previous_event_hash != previous or event.event_hash != expected_hash:\n                return {"valid": False, "broken_event_id": event.id, "event_count": len(events)}\n            previous = event.event_hash\n        return {"valid": True, "broken_event_id": None, "event_count": len(events)}\n\n    @app.get("/analysis-runs/{run_id}")\n    def get_analysis_run(run_id: str, session: Session = Depends(get_session)):\n        run = require(session, AnalysisRun, run_id, "analysis_run")\n        steps = list(session.scalars(select(AnalysisStep).where(AnalysisStep.analysis_run_id == run.id).order_by(AnalysisStep.sequence)))\n        return {"analysis_run_id": run.id, "sample_id": run.sample_id, "input_asset_id": run.input_asset_id,\n            "input_asset_checksum": run.input_asset_checksum, "pipeline_version": run.pipeline_version,\n            "status": run.status, "requested_models": run.requested_models_json, "model_versions": run.model_versions_json,\n            "geospatial_layer_versions": run.geospatial_layer_versions_json, "warnings": run.warnings_json,\n            "results": run.results_json, "started_at": run.started_at, "completed_at": run.completed_at,\n            "steps": [{"step_name": s.step_name, "sequence": s.sequence, "status": s.status, "version": s.version,\n                "output": s.output_json, "warning": s.warning_json, "error": s.error_json,\n                "started_at": s.started_at, "completed_at": s.completed_at} for s in steps]}\n\n    # Feature 9 catalog.\n    @app.get("/failure-codes")\n    def failure_codes():\n        from .services import ERROR_CATALOG\n        return ERROR_CATALOG\n\n    # Feature 10: model registry and cards. Compare route precedes /models/{id}.\n    @app.post("/models", response_model=ModelRegistryRead, status_code=201)\n    def create_model(payload: ModelRegistryCreate, session: Session = Depends(get_session)):\n        if payload.status in {"production_candidate", "production"} and not payload.dataset_version_id:\n            raise HTTPException(422, detail=issue("DATASET_VERSION_REQUIRED", "Production models must reference the exact frozen dataset used for training"))\n        if payload.dataset_version_id:\n            dataset = require(session, DatasetVersion, payload.dataset_version_id, "dataset_version")\n            if payload.status in {"production_candidate", "production"} and dataset.status != "frozen":\n                raise HTTPException(422, detail=issue("DATASET_NOT_FROZEN", "Production models must reference a frozen dataset"))\n        entry = ModelRegistryEntry(model_type=payload.model_type.strip().lower(), version=payload.version.strip(),\n            status=payload.status, dataset_version_id=payload.dataset_version_id, artifact_path=payload.artifact_path,\n            validation_status=payload.validation_status, card_json=payload.card, metrics_json=payload.metrics, threshold=payload.threshold)\n        session.add(entry)\n        try:\n            session.flush(); append_event(session, "MODEL_REGISTERED", "model", entry.id, payload={"model_type": entry.model_type, "version": entry.version}); session.commit()\n        except IntegrityError as exc:\n            session.rollback(); raise HTTPException(409, detail=issue("MODEL_VERSION_EXISTS", "This model type and version already exists")) from exc\n        session.refresh(entry); return entry\n\n    @app.get("/models", response_model=list[ModelRegistryRead])\n    def list_models(model_type: str | None = None, session: Session = Depends(get_session)):\n        query = select(ModelRegistryEntry)\n        if model_type: query = query.where(ModelRegistryEntry.model_type == model_type)\n        return list(session.scalars(query.order_by(ModelRegistryEntry.model_type, ModelRegistryEntry.created_at)))\n\n    @app.get("/models/compare")\n    def compare_models(model_a: str, model_b: str, session: Session = Depends(get_session)):\n        predictions_a = list(session.scalars(select(Prediction).where(Prediction.model_version == model_a)))\n        predictions_b = list(session.scalars(select(Prediction).where(Prediction.model_version == model_b)))\n        by_b = {p.sample_id: p for p in predictions_b}; comparisons = []\n        for first in predictions_a:\n            second = by_b.get(first.sample_id)\n            if not second: continue\n            ground = _ground_truth(session, first)\n            a_value = _primary_value(first); b_value = _primary_value(second)\n            row = {"sample_id": first.sample_id, "ground_truth_value": ground, "model_a_value": a_value, "model_b_value": b_value}\n            if isinstance(ground, (int, float)) and isinstance(a_value, (int, float)) and isinstance(b_value, (int, float)):\n                row.update({"model_a_absolute_error": abs(a_value-ground), "model_b_absolute_error": abs(b_value-ground),\n                    "result": "improved" if abs(b_value-ground) < abs(a_value-ground) else ("regressed" if abs(b_value-ground) > abs(a_value-ground) else "unchanged")})\n            comparisons.append(row)\n        return {"model_a": model_a, "model_b": model_b, "shared_sample_count": len(comparisons), "comparisons": comparisons}\n\n    @app.get("/models/{model_id}", response_model=ModelRegistryRead)\n    def get_model(model_id: str, session: Session = Depends(get_session)):\n        return require(session, ModelRegistryEntry, model_id, "model")\n\n    @app.patch("/models/{model_id}", response_model=ModelRegistryRead)\n    def update_model(model_id: str, payload: ModelRegistryUpdate, session: Session = Depends(get_session)):\n        entry = require(session, ModelRegistryEntry, model_id, "model")\n        values = payload.model_dump(exclude_unset=True)\n        allowed_statuses = {"development", "validation_candidate", "production_candidate", "production", "retired"}\n        if values.get("status") and values["status"] not in allowed_statuses: raise HTTPException(422, detail=issue("INVALID_MODEL_STATUS", "Invalid model status"))\n        target_status = values.get("status", entry.status)\n        target_dataset_id = values.get("dataset_version_id", entry.dataset_version_id)\n        if target_status in {"production_candidate", "production"}:\n            if not target_dataset_id:\n                raise HTTPException(422, detail=issue("DATASET_VERSION_REQUIRED", "Production models must reference a frozen dataset"))\n            dataset = require(session, DatasetVersion, target_dataset_id, "dataset_version")\n            if dataset.status != "frozen":\n                raise HTTPException(422, detail=issue("DATASET_NOT_FROZEN", "Production models must reference a frozen dataset"))\n        mapping = {"card": "card_json", "metrics": "metrics_json"}\n        for key, value in values.items(): setattr(entry, mapping.get(key, key), value)\n        append_event(session, "MODEL_UPDATED", "model", entry.id, payload={"changed_fields": sorted(values)})\n        session.commit(); session.refresh(entry); return entry\n\n    # Feature 11: active-learning queues.\n    @app.post("/active-learning/recalculate", response_model=list[ActiveLearningRead])\n    def recalculate_active_learning(payload: ActiveLearningRecalculate, session: Session = Depends(get_session)):\n        samples = list(session.scalars(select(Sample).order_by(Sample.created_at)))\n        growth_labels = [p.predicted_value_json.get("class") for p in session.scalars(select(Prediction).where(Prediction.model_type == "growth_form"))]\n        label_counts = Counter(label for label in growth_labels if label); max_count = max(label_counts.values(), default=1)\n        output = []\n        for sample in samples:\n            predictions = list(session.scalars(select(Prediction).where(Prediction.sample_id == sample.id)))\n            confidences = [p.confidence for p in predictions if p.confidence is not None]\n            model_uncertainty = 1 - (sum(confidences)/len(confidences) if confidences else .5)\n            area_count = session.scalar(select(func.count()).select_from(Sample).where(Sample.study_area_id == sample.study_area_id)) if sample.study_area_id else 1\n            geographic_uncertainty = 1 / max(1, area_count)\n            growth = next((p for p in reversed(predictions) if p.model_type == "growth_form"), None)\n            label = growth.predicted_value_json.get("class") if growth else None\n            class_rarity = 1 - (label_counts.get(label, 0) / max_count) if label else .5\n            quality = session.scalar(select(QualityAssessment).where(QualityAssessment.sample_id == sample.id).order_by(QualityAssessment.created_at.desc()))\n            sample_quality = quality.quality_score if quality else .5\n            study_importance = .8 if sample.site_type in {"school", "roadside"} else .5\n            components = {"model_uncertainty": model_uncertainty, "geographic_uncertainty": geographic_uncertainty,\n                "class_rarity": class_rarity, "sample_quality": sample_quality, "study_importance": study_importance}\n            base = sum(components[key] * payload.weights[key] for key in components)\n            reasons = []\n            if model_uncertainty >= .3: reasons.append("low_model_confidence")\n            if geographic_uncertainty >= .5: reasons.append("low_sample_density_zone")\n            if class_rarity >= .5: reasons.append("underrepresented_growth_form_class")\n            if not reasons: reasons.append("routine_diversity_review")\n            for queue_type, adjustment in (("annotation_review", model_uncertainty*.1), ("geographic_sampling", geographic_uncertainty*.1), ("class_balancing", class_rarity*.1)):\n                existing = session.scalar(select(ActiveLearningItem).where(ActiveLearningItem.sample_id == sample.id, ActiveLearningItem.queue_type == queue_type))\n                item = existing or ActiveLearningItem(sample_id=sample.id, queue_type=queue_type)\n                item.priority_score = round(min(1, base + adjustment), 4); item.priority_reasons_json = reasons\n                item.components_json = {k: round(v, 4) for k,v in components.items()}; item.formula_version = payload.formula_version\n                if not existing: item.status = "queued"; session.add(item)\n                output.append(item)\n        session.flush(); append_event(session, "ACTIVE_LEARNING_RECALCULATED", "active_learning", new_uuid(), payload={"formula_version": payload.formula_version, "sample_count": len(samples)})\n        session.commit(); return sorted(output, key=lambda item: item.priority_score, reverse=True)\n\n    @app.get("/active-learning/queue", response_model=list[ActiveLearningRead])\n    def active_learning_queue(queue_type: str | None = None, item_status: str | None = Query(default="queued", alias="status"), session: Session = Depends(get_session)):\n        query = select(ActiveLearningItem)\n        if queue_type: query = query.where(ActiveLearningItem.queue_type == queue_type)\n        if item_status: query = query.where(ActiveLearningItem.status == item_status)\n        return list(session.scalars(query.order_by(ActiveLearningItem.priority_score.desc(), ActiveLearningItem.updated_at)))\n\n    @app.patch("/active-learning/items/{item_id}", response_model=ActiveLearningRead)\n    def update_active_item(item_id: str, payload: ActiveLearningUpdate, session: Session = Depends(get_session)):\n        item = require(session, ActiveLearningItem, item_id, "active_learning_item"); item.status = payload.status\n        append_event(session, "ACTIVE_LEARNING_STATUS_CHANGED", "active_learning_item", item.id, sample_id=item.sample_id, payload={"status": item.status})\n        session.commit(); session.refresh(item); return item\n\n    # Feature 12: reviewer-dashboard data with filtering/sorting.\n    @app.get("/reviewer-dashboard")\n    def reviewer_dashboard(correction_type: str | None = None, model_type: str | None = None,\n            confidence_min: float | None = Query(default=None, ge=0, le=1), confidence_max: float | None = Query(default=None, ge=0, le=1),\n            study_area: str | None = None, date_from: datetime | None = None, date_to: datetime | None = None,\n            review_status: str | None = Query(default=None, alias="status"), sort: str = "priority_desc",\n            session: Session = Depends(get_session)):\n        query = select(Correction)\n        if correction_type: query = query.where(Correction.field_name == correction_type)\n        if review_status: query = query.where(Correction.status == review_status)\n        rows = []\n        for correction in session.scalars(query):\n            prediction = require(session, Prediction, correction.prediction_id, "prediction"); sample = require(session, Sample, correction.sample_id, "sample")\n            if model_type and prediction.model_type != model_type: continue\n            if confidence_min is not None and (prediction.confidence is None or prediction.confidence < confidence_min): continue\n            if confidence_max is not None and (prediction.confidence is None or prediction.confidence > confidence_max): continue\n            if study_area and sample.study_area_id != study_area: continue\n            if date_from and correction.created_at.replace(tzinfo=date_from.tzinfo) < date_from: continue\n            if date_to and correction.created_at.replace(tzinfo=date_to.tzinfo) > date_to: continue\n            image = session.scalar(select(SampleAsset).where(SampleAsset.sample_id == sample.id, SampleAsset.asset_type == "original_image").order_by(SampleAsset.created_at))\n            priority_item = session.scalar(select(ActiveLearningItem).where(ActiveLearningItem.sample_id == sample.id, ActiveLearningItem.queue_type == "annotation_review"))\n            priority = priority_item.priority_score if priority_item else round(1-(prediction.confidence or .5), 4)\n            rows.append({"correction_id": correction.id, "sample_id": sample.id,\n                "original_image_url": f"/assets/{image.id}/content" if image else None,\n                "original_prediction": {"model_type": prediction.model_type, "value": correction.original_value_json, "confidence": prediction.confidence},\n                "proposed_correction": {"field": correction.field_name, "value": correction.proposed_value_json, "asset_id": correction.proposed_asset_id},\n                "reason": correction.reason, "priority": priority, "status": correction.status, "study_area_id": sample.study_area_id})\n        reverse = sort != "priority_asc"; rows.sort(key=lambda row: row["priority"], reverse=reverse)\n        return {"items": rows, "count": len(rows), "sort": sort}\n\n    # Feature 13: immutable reanalysis and model comparisons.\n    @app.post("/samples/{sample_id}/reanalyze", response_model=PredictionRead, status_code=201)\n    def reanalyze(sample_id: str, payload: ReanalysisRequest, session: Session = Depends(get_session)):\n        require(session, Sample, sample_id, "sample"); source = require(session, SampleAsset, payload.asset_id, "asset")\n        if source.sample_id != sample_id: raise HTTPException(422, detail=issue("ASSET_SAMPLE_MISMATCH", "Asset belongs to another sample"))\n        prediction = run_adapter(payload.model_type, source, session, requested_version=payload.model_version)\n        append_event(session, "SAMPLE_REANALYZED", "sample", sample_id, sample_id=sample_id, payload={"prediction_id": prediction.id, "model_version": prediction.model_version})\n        session.commit(); session.refresh(prediction); return prediction\n\n    @app.get("/samples/{sample_id}/model-comparison")\n    def sample_model_comparison(sample_id: str, session: Session = Depends(get_session)):\n        require(session, Sample, sample_id, "sample"); predictions = list(session.scalars(select(Prediction).where(Prediction.sample_id == sample_id).order_by(Prediction.created_at)))\n        groups = []\n        for prediction in predictions:\n            ground = _ground_truth(session, prediction); value = _primary_value(prediction)\n            groups.append({"prediction_id": prediction.id, "model_type": prediction.model_type, "model_version": prediction.model_version,\n                "value": value, "confidence": prediction.confidence, "ground_truth_value": ground,\n                "absolute_error": abs(value-ground) if isinstance(value,(int,float)) and isinstance(ground,(int,float)) else None})\n        return {"sample_id": sample_id, "predictions": groups}\n\n    # Feature 14: configurable image quality gates.\n    @app.post("/assets/{asset_id}/quality", response_model=QualityAssessmentRead, status_code=201)\n    def assess_quality(asset_id: str, session: Session = Depends(get_session)):\n        asset = require(session, SampleAsset, asset_id, "asset"); sample = require(session, Sample, asset.sample_id, "sample")\n        metadata_complete = all([sample.substrate_type, sample.site_type, sample.latitude is not None, sample.captured_at])\n        result = quality_service.assess(asset_path(asset), metadata_complete=metadata_complete)\n        row = QualityAssessment(sample_id=sample.id, asset_id=asset.id, quality_score=result["quality_score"], decision=result["decision"],\n            metrics_json=result["metrics"], warnings_json=result["warnings"], thresholds_json=result["thresholds"], quality_version=result["quality_version"])\n        session.add(row); session.flush(); append_event(session, "IMAGE_QUALITY_ASSESSED", "quality_assessment", row.id, sample_id=sample.id,\n            payload={"asset_id": asset.id, "decision": row.decision, "quality_score": row.quality_score}); session.commit(); session.refresh(row); return row\n\n    @app.get("/samples/{sample_id}/quality", response_model=list[QualityAssessmentRead])\n    def sample_quality(sample_id: str, session: Session = Depends(get_session)):\n        require(session, Sample, sample_id, "sample"); return list(session.scalars(select(QualityAssessment).where(QualityAssessment.sample_id == sample_id).order_by(QualityAssessment.created_at)))\n\n    # Features 15 and 16: orchestration plus decomposed confidence.\n    @app.post("/analyze-sample")\n    def analyze_sample(payload: AnalyzeSampleRequest, session: Session = Depends(get_session)):\n        sample = require(session, Sample, payload.sample_id, "sample"); source = require(session, SampleAsset, payload.asset_id, "asset")\n        if source.sample_id != sample.id or source.asset_type != "original_image": raise HTTPException(422, detail=issue("INVALID_ANALYSIS_ASSET", "Analysis requires the sample\'s original image"))\n        run = AnalysisRun(sample_id=sample.id, input_asset_id=source.id, input_asset_checksum=source.checksum,\n            pipeline_version=payload.pipeline_version, status="running", requested_models_json=payload.run_models,\n            model_versions_json={}, geospatial_layer_versions_json=payload.geospatial_layer_versions, warnings_json=[], results_json={})\n        session.add(run); session.flush(); append_event(session, "ANALYSIS_STARTED", "analysis_run", run.id, sample_id=sample.id, payload={"asset_id": source.id, "models": payload.run_models})\n        quality_result = quality_service.assess(asset_path(source), metadata_complete=all([sample.site_type, sample.substrate_type, sample.latitude is not None]))\n        quality_row = QualityAssessment(sample_id=sample.id, asset_id=source.id, quality_score=quality_result["quality_score"], decision=quality_result["decision"],\n            metrics_json=quality_result["metrics"], warnings_json=quality_result["warnings"], thresholds_json=quality_result["thresholds"], quality_version=quality_result["quality_version"])\n        session.add(quality_row); session.flush()\n        quality_step = AnalysisStep(analysis_run_id=run.id, sequence=0, step_name="image_quality", status="completed", version=quality_result["quality_version"],\n            output_json=quality_result, started_at=run.started_at, completed_at=utc_now()); session.add(quality_step)\n        results: dict[str, Any] = {"image_quality": quality_result}; warnings: list[dict[str, Any]] = []\n        if quality_result["decision"] == "rejected":\n            warnings.extend(issue(code, f"Quality gate blocked analysis: {code}", details=quality_result["metrics"], status="needs_retake") for code in quality_result["warnings"])\n            run.status = "needs_retake"; run.results_json = results; run.warnings_json = warnings; run.completed_at = utc_now()\n            confidence = _store_confidence(session, sample.id, run.id, {"image_quality": quality_result["quality_score"]})\n            append_event(session, "ANALYSIS_COMPLETED", "analysis_run", run.id, sample_id=sample.id, payload={"status": run.status}); session.commit()\n            return _run_response(run, confidence)\n        for sequence, model_type in enumerate(payload.run_models, start=1):\n            step = AnalysisStep(analysis_run_id=run.id, sequence=sequence, step_name=model_type, status="running", started_at=utc_now()); session.add(step); session.flush()\n            try:\n                prediction = run_adapter(model_type, source, session, context=results)\n                step.status = "completed"; step.version = prediction.model_version; step.output_json = prediction.predicted_value_json; step.completed_at = utc_now()\n                results[model_type] = {**prediction.predicted_value_json, "confidence": prediction.confidence, "model_version": prediction.model_version,\n                    "prediction_id": prediction.id, "is_placeholder": prediction.is_placeholder}\n                run.model_versions_json = {**run.model_versions_json, model_type: prediction.model_version}\n                if prediction.is_placeholder:\n                    warning = issue("MODEL_PLACEHOLDER", f"{model_type} is using a development placeholder", status="warning")\n                    warnings.append(warning); step.warning_json = warning\n            except HTTPException as exc:\n                warning = exc.detail if isinstance(exc.detail, dict) else issue("MODEL_INTERNAL_ERROR", str(exc.detail))\n                if warning.get("severity") == "error": warning = {**warning, "severity": "warning", "status": "continued_with_warning"}\n                warnings.append(warning); step.status = "failed_nonblocking"; step.error_json = warning; step.completed_at = utc_now()\n        confidence_components = {"image_quality": quality_result["quality_score"],\n            "coverage": results.get("coverage", {}).get("confidence"), "growth_form": results.get("growth_form", {}).get("confidence"),\n            "chlorosis": results.get("chlorosis", {}).get("confidence"),\n            "geospatial_completeness": (1.0 if sample.latitude is not None else 0.0),\n            "risk_engine": results.get("environmental_risk", {}).get("confidence"),\n            "feature_completeness": len([m for m in payload.run_models if m in results]) / len(payload.run_models)}\n        confidence = _store_confidence(session, sample.id, run.id, confidence_components)\n        run.status = "completed_with_warnings" if warnings else "completed"; run.results_json = results; run.warnings_json = warnings; run.completed_at = utc_now()\n        append_event(session, "ANALYSIS_COMPLETED", "analysis_run", run.id, sample_id=sample.id,\n            payload={"status": run.status, "model_versions": run.model_versions_json, "confidence_assessment_id": confidence.id})\n        session.commit(); return _run_response(run, confidence)\n\n    def _store_confidence(session: Session, sample_id: str, run_id: str | None, components: dict[str, float | None],\n                          version: str = "sample_confidence_v0", weights: dict[str,float] | None = None) -> ConfidenceAssessment:\n        complete = {key: components.get(key) for key in ("image_quality", "coverage", "growth_form", "chlorosis", "geospatial_completeness", "risk_engine", "feature_completeness")}\n        result = aggregate_confidence(complete, weights)\n        row = ConfidenceAssessment(sample_id=sample_id, analysis_run_id=run_id, components_json=complete,\n            overall_confidence=result["overall_confidence"], weights_json=result["weights"], limitations_json=result["limitations"], confidence_version=version)\n        session.add(row); session.flush(); return row\n\n    def _run_response(run: AnalysisRun, confidence: ConfidenceAssessment) -> dict[str, Any]:\n        return {"analysis_run_id": run.id, "status": run.status, "results": run.results_json,\n            "warnings": run.warnings_json, "model_versions": run.model_versions_json,\n            "confidence": ConfidenceRead.model_validate(confidence).model_dump(mode="json")}\n\n    @app.post("/samples/{sample_id}/confidence", response_model=ConfidenceRead, status_code=201)\n    def create_confidence(sample_id: str, payload: ConfidenceCreate, session: Session = Depends(get_session)):\n        require(session, Sample, sample_id, "sample")\n        if payload.analysis_run_id:\n            run = require(session, AnalysisRun, payload.analysis_run_id, "analysis_run")\n            if run.sample_id != sample_id: raise HTTPException(422, detail=issue("RUN_SAMPLE_MISMATCH", "Analysis run belongs to another sample"))\n        try: row = _store_confidence(session, sample_id, payload.analysis_run_id, payload.components, payload.confidence_version, payload.weights)\n        except ValueError as exc: raise HTTPException(422, detail=issue("INVALID_CONFIDENCE_COMPONENTS", str(exc))) from exc\n        append_event(session, "CONFIDENCE_CALCULATED", "confidence_assessment", row.id, sample_id=sample_id,\n            payload={"overall_confidence": row.overall_confidence, "confidence_version": row.confidence_version}); session.commit(); session.refresh(row); return row\n\n    @app.get("/samples/{sample_id}/confidence", response_model=list[ConfidenceRead])\n    def list_confidence(sample_id: str, session: Session = Depends(get_session)):\n        require(session, Sample, sample_id, "sample"); return list(session.scalars(select(ConfidenceAssessment).where(ConfidenceAssessment.sample_id == sample_id).order_by(ConfidenceAssessment.created_at)))\n\n\ndef _primary_value(prediction: Prediction) -> Any:\n    data = prediction.predicted_value_json or {}\n    keys = {"coverage": "coverage_percent", "growth_form": "class", "chlorosis": "severity", "environmental_risk": "score"}\n    return data.get(keys.get(prediction.model_type, "value"))\n\n\ndef _ground_truth(session: Session, prediction: Prediction) -> Any:\n    field = {"coverage": "coverage_percent", "growth_form": "growth_form", "chlorosis": "chlorosis_severity"}.get(prediction.model_type)\n    if not field: return None\n    staging = session.scalar(select(DatasetStagingItem).where(DatasetStagingItem.sample_id == prediction.sample_id,\n        DatasetStagingItem.field_name == field).order_by(DatasetStagingItem.staged_at.desc()))\n    return staging.verified_value_json if staging else None\n', 'app/main.py': '"""EcoSignal standalone backend through Feature 16."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Generator\nfrom contextlib import asynccontextmanager\nfrom hashlib import sha256\nfrom io import BytesIO\nimport os\nfrom pathlib import Path\nimport re\nfrom tempfile import NamedTemporaryFile\n\nfrom fastapi import Depends, FastAPI, File, Form, HTTPException, Query, Request, Response, UploadFile, status\nfrom fastapi.exceptions import RequestValidationError\nfrom fastapi.encoders import jsonable_encoder\nfrom fastapi.responses import FileResponse, JSONResponse\nfrom PIL import Image, UnidentifiedImageError\nfrom sqlalchemy import inspect, select, text\nfrom sqlalchemy.exc import IntegrityError\nfrom sqlalchemy.orm import Session\n\nfrom .database import Base, build_engine, build_session_factory, session_dependency\nfrom .inference import CoverageModel, get_coverage_model\nfrom .models import (\n    Correction,\n    DatasetStagingItem,\n    DatasetVersion,\n    ModelRegistryEntry,\n    Prediction,\n    Review,\n    Sample,\n    SampleAsset,\n    new_uuid,\n    utc_now,\n)\nfrom .remaining import register_remaining_routes\nfrom .schemas import (\n    AssetRead,\n    CoverageAnalysisRequest,\n    CorrectionCreate,\n    CorrectionRead,\n    CorrectionWithdrawRequest,\n    DatasetStagingRead,\n    PredictionCreate,\n    PredictionRead,\n    ReviewDecisionCreate,\n    ReviewDecisionResult,\n    ReviewDetail,\n    ReviewQueueItem,\n    ReviewRead,\n    SampleCreate,\n    SampleRead,\n    SampleUpdate,\n)\nfrom .services import QualityService, append_event, issue\n\nMAX_ASSET_BYTES = 20 * 1024 * 1024\nSUPPORTED_IMAGE_FORMATS = {\n    "JPEG": ("image/jpeg", ".jpg"),\n    "PNG": ("image/png", ".png"),\n    "WEBP": ("image/webp", ".webp"),\n}\n\n\ndef migrate_feature_1_asset_table(engine) -> None:\n    """Add Feature 2 columns when reusing an existing Feature 1 SQLite file."""\n    inspector = inspect(engine)\n    if "sample_assets" not in inspector.get_table_names():\n        return\n    existing = {column["name"] for column in inspector.get_columns("sample_assets")}\n    additions = {\n        "width": "INTEGER",\n        "height": "INTEGER",\n        "created_by": "VARCHAR(128)",\n        "source_asset_id": "VARCHAR(36)",\n        "original_filename": "VARCHAR(255)",\n    }\n    with engine.begin() as connection:\n        for column, sql_type in additions.items():\n            if column not in existing:\n                connection.execute(text(f"ALTER TABLE sample_assets ADD COLUMN {column} {sql_type}"))\n        connection.execute(text(\n            "CREATE UNIQUE INDEX IF NOT EXISTS uq_asset_sample_checksum "\n            "ON sample_assets (sample_id, checksum)"\n        ))\n        connection.execute(text(\n            "CREATE INDEX IF NOT EXISTS ix_sample_assets_source_asset_id "\n            "ON sample_assets (source_asset_id)"\n        ))\n    if "reviews" in inspector.get_table_names():\n        review_columns = {column["name"] for column in inspector.get_columns("reviews")}\n        with engine.begin() as connection:\n            if "reviewed_value_json" not in review_columns:\n                connection.execute(text("ALTER TABLE reviews ADD COLUMN reviewed_value_json JSON"))\n            if "assigned_dataset_version_id" not in review_columns:\n                connection.execute(text("ALTER TABLE reviews ADD COLUMN assigned_dataset_version_id VARCHAR(36)"))\n\n\ndef inspect_image(data: bytes) -> tuple[str, str, int, int]:\n    try:\n        with Image.open(BytesIO(data)) as image:\n            image.verify()\n        with Image.open(BytesIO(data)) as image:\n            image_format = (image.format or "").upper()\n            width, height = image.size\n    except (UnidentifiedImageError, OSError, SyntaxError, Image.DecompressionBombError) as exc:\n        raise HTTPException(status_code=422, detail="The uploaded file is not a valid image") from exc\n    if image_format not in SUPPORTED_IMAGE_FORMATS:\n        raise HTTPException(status_code=415, detail="Unsupported image type. Use JPEG, PNG, or WEBP.")\n    mime_type, extension = SUPPORTED_IMAGE_FORMATS[image_format]\n    return mime_type, extension, width, height\n\n\ndef safe_asset_type(value: str) -> str:\n    cleaned = re.sub(r"[^a-z0-9_]+", "_", value.strip().lower()).strip("_")\n    if not cleaned or len(cleaned) > 64:\n        raise HTTPException(status_code=422, detail="asset_type must be 1-64 letters, numbers, or underscores")\n    return cleaned\n\n\ndef create_app(\n    database_url: str | None = None,\n    storage_root: str | Path | None = None,\n    coverage_model: CoverageModel | None = None,\n    model_adapters: dict[str, object] | None = None,\n    quality_thresholds: dict[str, float] | None = None,\n) -> FastAPI:\n    engine = build_engine(database_url)\n    session_factory = build_session_factory(engine)\n    asset_root = Path(storage_root or os.getenv("ECOSIGNAL_ASSET_ROOT", "./ecosignal_assets")).resolve()\n    selected_coverage_model = coverage_model or get_coverage_model()\n    selected_adapters = dict(model_adapters or {})\n    selected_adapters.setdefault("coverage", selected_coverage_model)\n    quality_service = QualityService(quality_thresholds)\n\n    @asynccontextmanager\n    async def lifespan(_: FastAPI):\n        Base.metadata.create_all(engine)\n        migrate_feature_1_asset_table(engine)\n        asset_root.mkdir(parents=True, exist_ok=True)\n        with session_factory() as bootstrap_session:\n            for model_type, adapter in selected_adapters.items():\n                version = str(getattr(adapter, "model_version", "unversioned_adapter"))\n                existing = bootstrap_session.scalar(select(ModelRegistryEntry).where(\n                    ModelRegistryEntry.model_type == model_type,\n                    ModelRegistryEntry.version == version,\n                ))\n                if existing is None:\n                    placeholder = bool(getattr(adapter, "is_placeholder", "placeholder" in version.lower()))\n                    bootstrap_session.add(ModelRegistryEntry(\n                        model_type=model_type, version=version,\n                        status="development" if placeholder else "validation_candidate",\n                        validation_status="pending" if placeholder else "partial",\n                        card_json={\n                            "name": version, "purpose": f"{model_type} adapter connected to EcoSignal",\n                            "input": "Validated sample image and pipeline context", "output": f"{model_type} prediction JSON",\n                            "training_data": "Not recorded by adapter bootstrap", "validation_data": "Pending registry update",\n                            "known_failure_cases": ["Adapter-specific validation is incomplete"],\n                            "intended_use": "EcoSignal development and validation workflow",\n                            "prohibited_use": "Do not treat unvalidated outputs as calibrated scientific measurements",\n                        }, metrics_json={}, threshold=None,\n                    ))\n            bootstrap_session.commit()\n        yield\n        engine.dispose()\n\n    app = FastAPI(\n        title="EcoSignal Standalone Backend",\n        version="1.0.0-features-1-16",\n        description="Complete model-independent EcoSignal scientific workflow through Feature 16.",\n        lifespan=lifespan,\n    )\n    app.state.engine = engine\n    app.state.session_factory = session_factory\n    app.state.asset_root = asset_root\n    app.state.coverage_model = selected_coverage_model\n    app.state.model_adapters = selected_adapters\n    app.state.quality_service = quality_service\n\n    @app.exception_handler(HTTPException)\n    async def standardized_http_error(_: Request, exc: HTTPException):\n        if isinstance(exc.detail, dict) and {"code", "message", "severity"} <= set(exc.detail):\n            body = {**exc.detail, "detail": exc.detail}\n        else:\n            code = {404: "RESOURCE_NOT_FOUND", 409: "CONFLICT", 413: "ASSET_TOO_LARGE", 415: "UNSUPPORTED_FILE_TYPE", 422: "VALIDATION_ERROR", 500: "INTERNAL_ERROR", 503: "SERVICE_UNAVAILABLE"}.get(exc.status_code, "REQUEST_FAILED")\n            body = {**issue(code, str(exc.detail)), "detail": exc.detail}\n        return JSONResponse(status_code=exc.status_code, content=body, headers=exc.headers)\n\n    @app.exception_handler(RequestValidationError)\n    async def standardized_validation_error(_: Request, exc: RequestValidationError):\n        details = {"errors": jsonable_encoder(exc.errors())}\n        body = issue("VALIDATION_ERROR", "Request validation failed", details=details)\n        body["detail"] = details["errors"]\n        return JSONResponse(status_code=422, content=body)\n\n    def get_session() -> Generator[Session, None, None]:\n        yield from session_dependency(session_factory)\n\n    def ensure_model_entry(model_type: str, model_version: str, is_placeholder: bool, session: Session) -> ModelRegistryEntry:\n        entry = session.scalar(select(ModelRegistryEntry).where(\n            ModelRegistryEntry.model_type == model_type,\n            ModelRegistryEntry.version == model_version,\n        ))\n        if entry is None:\n            entry = ModelRegistryEntry(\n                model_type=model_type, version=model_version,\n                status="development" if is_placeholder else "validation_candidate",\n                validation_status="pending" if is_placeholder else "partial",\n                card_json={\n                    "name": model_version, "purpose": f"Externally attached {model_type} prediction source",\n                    "input": "EcoSignal sample evidence", "output": f"{model_type} prediction JSON",\n                    "training_data": "Not supplied at attachment time", "validation_data": "Not supplied at attachment time",\n                    "known_failure_cases": ["Validation metadata not yet completed"],\n                    "intended_use": "Traceable EcoSignal development record",\n                    "prohibited_use": "Do not claim calibrated scientific performance until the model card is completed",\n                }, metrics_json={}, threshold=None,\n            )\n            session.add(entry); session.flush()\n        return entry\n\n    def require_sample(sample_id: str, session: Session) -> Sample:\n        sample = session.get(Sample, sample_id)\n        if sample is None:\n            raise HTTPException(status_code=404, detail="Sample not found")\n        return sample\n\n    def require_asset(asset_id: str, session: Session) -> SampleAsset:\n        asset = session.get(SampleAsset, asset_id)\n        if asset is None:\n            raise HTTPException(status_code=404, detail="Asset not found")\n        return asset\n\n    def require_prediction(prediction_id: str, session: Session) -> Prediction:\n        prediction = session.get(Prediction, prediction_id)\n        if prediction is None:\n            raise HTTPException(status_code=404, detail="Prediction not found")\n        return prediction\n\n    def require_correction(correction_id: str, session: Session) -> Correction:\n        correction = session.get(Correction, correction_id)\n        if correction is None:\n            raise HTTPException(status_code=404, detail="Correction not found")\n        return correction\n\n    def approvals_required(correction: Correction) -> int:\n        return 2 if correction.field_name in {"coverage_mask", "chlorosis_mask"} else 1\n\n    def approved_reviews(correction_id: str, session: Session) -> list[Review]:\n        statement = (\n            select(Review)\n            .where(Review.correction_id == correction_id, Review.decision == "approved")\n            .order_by(Review.reviewed_at, Review.id)\n        )\n        return list(session.scalars(statement))\n\n    def review_detail(correction: Correction, session: Session) -> ReviewDetail:\n        prediction = require_prediction(correction.prediction_id, session)\n        sample = require_sample(correction.sample_id, session)\n        assets = list(session.scalars(\n            select(SampleAsset)\n            .where(SampleAsset.sample_id == correction.sample_id)\n            .order_by(SampleAsset.created_at, SampleAsset.id)\n        ))\n        history = list(session.scalars(\n            select(Correction)\n            .where(Correction.sample_id == correction.sample_id, Correction.id != correction.id)\n            .order_by(Correction.created_at, Correction.id)\n        ))\n        reviews = list(session.scalars(\n            select(Review)\n            .where(Review.correction_id == correction.id)\n            .order_by(Review.reviewed_at, Review.id)\n        ))\n        staging = session.scalar(\n            select(DatasetStagingItem).where(DatasetStagingItem.correction_id == correction.id)\n        )\n        approved_count = sum(item.decision == "approved" for item in reviews)\n        requirements: dict[str, object] = {\n            "approvals_required": approvals_required(correction),\n            "approvals_received": approved_count,\n            "independent_reviewer_required": True,\n            "detailed_note_required": False,\n            "warnings": [],\n        }\n        if correction.field_name in {"coverage_mask", "chlorosis_mask", "growth_form"}:\n            requirements["detailed_note_required"] = True\n        if correction.field_name == "coverage_percent":\n            try:\n                delta = abs(float(correction.proposed_value_json) - float(correction.original_value_json))\n            except (TypeError, ValueError):\n                delta = 0.0\n            requirements["coverage_change_percentage_points"] = delta\n            if delta > 20:\n                requirements["detailed_note_required"] = True\n                requirements["warnings"] = ["LARGE_COVERAGE_CHANGE_REQUIRES_DETAILED_NOTE"]\n        return ReviewDetail(\n            correction=CorrectionRead.model_validate(correction),\n            prediction=PredictionRead.model_validate(prediction),\n            sample=SampleRead.model_validate(sample),\n            assets=[AssetRead.model_validate(asset) for asset in assets],\n            previous_corrections=[CorrectionRead.model_validate(item) for item in history],\n            reviews=[ReviewRead.model_validate(item) for item in reviews],\n            staging_item=DatasetStagingRead.model_validate(staging) if staging else None,\n            review_requirements=requirements,\n        )\n\n    @app.get("/health")\n    def health() -> dict[str, str]:\n        return {\n            "status": "ok",\n            "feature": "complete_features_1_through_16",\n            "coverage_model": selected_coverage_model.model_version,\n        }\n\n    @app.post("/samples", response_model=SampleRead, status_code=status.HTTP_201_CREATED)\n    def create_sample(payload: SampleCreate, session: Session = Depends(get_session)):\n        sample = Sample(**payload.model_dump())\n        session.add(sample)\n        session.flush()\n        append_event(session, "SAMPLE_CREATED", "sample", sample.id, sample_id=sample.id,\n            actor_id=sample.user_id, payload={"study_area_id": sample.study_area_id, "status": sample.status})\n        session.commit()\n        session.refresh(sample)\n        return sample\n\n    @app.get("/samples/{sample_id}", response_model=SampleRead)\n    def get_sample(sample_id: str, session: Session = Depends(get_session)):\n        return require_sample(sample_id, session)\n\n    @app.patch("/samples/{sample_id}", response_model=SampleRead)\n    def update_sample(sample_id: str, payload: SampleUpdate, session: Session = Depends(get_session)):\n        sample = require_sample(sample_id, session)\n        for field, value in payload.model_dump(exclude_unset=True).items():\n            setattr(sample, field, value)\n        session.commit()\n        session.refresh(sample)\n        return sample\n\n    @app.get("/samples/{sample_id}/predictions", response_model=list[PredictionRead])\n    def list_predictions(sample_id: str, session: Session = Depends(get_session)):\n        require_sample(sample_id, session)\n        statement = select(Prediction).where(Prediction.sample_id == sample_id).order_by(Prediction.created_at, Prediction.id)\n        return list(session.scalars(statement))\n\n    @app.post("/samples/{sample_id}/predictions", response_model=PredictionRead, status_code=status.HTTP_201_CREATED)\n    def attach_prediction(sample_id: str, payload: PredictionCreate, session: Session = Depends(get_session)):\n        require_sample(sample_id, session)\n        data = payload.model_dump()\n        prediction = Prediction(\n            sample_id=sample_id,\n            model_type=data["model_type"],\n            model_version=data["model_version"],\n            predicted_value_json=data["predicted_value"],\n            confidence=data["confidence"],\n            status=data["status"],\n            warnings_json=data["warnings"],\n            is_placeholder=data["is_placeholder"],\n        )\n        ensure_model_entry(prediction.model_type, prediction.model_version, prediction.is_placeholder, session)\n        session.add(prediction)\n        session.flush()\n        append_event(session, "PREDICTION_CREATED", "prediction", prediction.id, sample_id=sample_id,\n            payload={"model_type": prediction.model_type, "model_version": prediction.model_version})\n        session.commit()\n        session.refresh(prediction)\n        return prediction\n\n    @app.post(\n        "/predictions/{prediction_id}/corrections",\n        response_model=CorrectionRead,\n        status_code=status.HTTP_201_CREATED,\n    )\n    def submit_correction(\n        prediction_id: str,\n        payload: CorrectionCreate,\n        session: Session = Depends(get_session),\n    ):\n        prediction = require_prediction(prediction_id, session)\n        field_name = payload.field_name.strip().lower()\n        output = prediction.predicted_value_json or {}\n\n        simple_fields = {\n            "coverage_percent": ("coverage", "coverage_percent"),\n            "growth_form": ("growth_form", "class"),\n            "chlorosis_severity": ("chlorosis", "severity"),\n            "image_quality_assessment": ("image_quality", "assessment"),\n        }\n        mask_fields = {\n            "coverage_mask": ("coverage", "mask_asset_id"),\n            "chlorosis_mask": ("chlorosis", "mask_asset_id"),\n        }\n        metadata_fields = {"substrate_type", "site_type", "notes", "captured_at"}\n\n        if field_name in simple_fields:\n            required_model, output_key = simple_fields[field_name]\n            if prediction.model_type != required_model:\n                raise HTTPException(status_code=422, detail=f"{field_name} does not match this prediction type")\n            if output_key not in output:\n                raise HTTPException(status_code=422, detail=f"Original {field_name} is unavailable")\n            if payload.proposed_value is None or payload.proposed_asset_id is not None:\n                raise HTTPException(status_code=422, detail="This correction requires proposed_value only")\n            original_value = output[output_key]\n            proposed_value = payload.proposed_value\n            if field_name == "coverage_percent":\n                if isinstance(proposed_value, bool) or not isinstance(proposed_value, (int, float)) or not 0 <= proposed_value <= 100:\n                    raise HTTPException(status_code=422, detail="coverage_percent must be from 0 to 100")\n                proposed_value = float(proposed_value)\n            elif field_name == "growth_form":\n                allowed = {"crustose", "foliose", "fruticose", "uncertain"}\n                if not isinstance(proposed_value, str) or proposed_value.strip().lower() not in allowed:\n                    raise HTTPException(status_code=422, detail="Invalid growth-form label")\n                proposed_value = proposed_value.strip().lower()\n            elif field_name == "chlorosis_severity":\n                allowed_labels = {"none", "mild", "moderate", "severe", "uncertain"}\n                numeric = isinstance(proposed_value, (int, float)) and not isinstance(proposed_value, bool)\n                labeled = isinstance(proposed_value, str) and proposed_value.strip().lower() in allowed_labels\n                if not ((numeric and 0 <= float(proposed_value) <= 1) or labeled):\n                    raise HTTPException(status_code=422, detail="Invalid chlorosis severity")\n                proposed_value = float(proposed_value) if numeric else proposed_value.strip().lower()\n        elif field_name in mask_fields:\n            required_model, output_key = mask_fields[field_name]\n            if prediction.model_type != required_model:\n                raise HTTPException(status_code=422, detail=f"{field_name} does not match this prediction type")\n            if payload.proposed_asset_id is None or payload.proposed_value is not None:\n                raise HTTPException(status_code=422, detail="Mask corrections require proposed_asset_id only")\n            proposed_asset = require_asset(payload.proposed_asset_id, session)\n            if proposed_asset.sample_id != prediction.sample_id:\n                raise HTTPException(status_code=422, detail="Corrected asset must belong to the prediction\'s sample")\n            if proposed_asset.asset_type not in {"reviewed_mask", "final_annotation"}:\n                raise HTTPException(status_code=422, detail="Use a reviewed_mask or final_annotation asset")\n            original_value = output.get(output_key)\n            proposed_value = None\n        elif field_name.startswith("sample_metadata."):\n            metadata_field = field_name.split(".", 1)[1]\n            if metadata_field not in metadata_fields:\n                raise HTTPException(status_code=422, detail="This sample metadata field is not user-correctable")\n            if payload.proposed_value is None or payload.proposed_asset_id is not None:\n                raise HTTPException(status_code=422, detail="Metadata corrections require proposed_value only")\n            sample = require_sample(prediction.sample_id, session)\n            original_value = getattr(sample, metadata_field)\n            if hasattr(original_value, "isoformat"):\n                original_value = original_value.isoformat()\n            proposed_value = payload.proposed_value\n        else:\n            raise HTTPException(\n                status_code=422,\n                detail="Field is not correctable. Risk, confidence, geospatial, and geomorphology outputs are protected.",\n            )\n\n        if proposed_value == original_value and payload.proposed_asset_id is None:\n            raise HTTPException(status_code=422, detail="Proposed value must differ from the original")\n        correction = Correction(\n            sample_id=prediction.sample_id,\n            prediction_id=prediction.id,\n            field_name=field_name,\n            original_value_json=original_value,\n            proposed_value_json=proposed_value,\n            proposed_asset_id=payload.proposed_asset_id,\n            correction_method=payload.correction_method,\n            reason=payload.reason.strip(),\n            submitted_by=payload.submitted_by.strip(),\n            status="pending_review",\n        )\n        session.add(correction)\n        session.flush()\n        append_event(session, "CORRECTION_SUBMITTED", "correction", correction.id,\n            sample_id=correction.sample_id, actor_id=correction.submitted_by,\n            payload={"prediction_id": correction.prediction_id, "field_name": correction.field_name})\n        session.commit()\n        session.refresh(correction)\n        return correction\n\n    @app.get("/corrections", response_model=list[CorrectionRead])\n    def list_corrections(\n        correction_status: str | None = Query(default=None, alias="status"),\n        sample_id: str | None = None,\n        session: Session = Depends(get_session),\n    ):\n        statement = select(Correction)\n        if correction_status is not None:\n            statement = statement.where(Correction.status == correction_status)\n        if sample_id is not None:\n            require_sample(sample_id, session)\n            statement = statement.where(Correction.sample_id == sample_id)\n        statement = statement.order_by(Correction.created_at, Correction.id)\n        return list(session.scalars(statement))\n\n    @app.get("/corrections/{correction_id}", response_model=CorrectionRead)\n    def get_correction(correction_id: str, session: Session = Depends(get_session)):\n        return require_correction(correction_id, session)\n\n    @app.patch("/corrections/{correction_id}/withdraw", response_model=CorrectionRead)\n    def withdraw_correction(\n        correction_id: str,\n        payload: CorrectionWithdrawRequest,\n        session: Session = Depends(get_session),\n    ):\n        correction = require_correction(correction_id, session)\n        if correction.submitted_by != payload.requested_by.strip():\n            raise HTTPException(status_code=403, detail="Only the submitter can withdraw this correction")\n        if correction.status in {"approved", "rejected", "withdrawn"}:\n            raise HTTPException(status_code=409, detail=f"Cannot withdraw a {correction.status} correction")\n        correction.status = "withdrawn"\n        correction.withdrawn_at = utc_now()\n        append_event(session, "CORRECTION_WITHDRAWN", "correction", correction.id,\n            sample_id=correction.sample_id, actor_id=payload.requested_by.strip())\n        session.commit()\n        session.refresh(correction)\n        return correction\n\n    @app.delete("/corrections/{correction_id}", include_in_schema=False)\n    def immutable_corrections_guard(correction_id: str):\n        del correction_id\n        raise HTTPException(status_code=405, detail="Corrections are audit records and cannot be deleted")\n\n    @app.get("/reviews/queue", response_model=list[ReviewQueueItem])\n    def get_review_queue(\n        review_status: str | None = Query(default=None, alias="status"),\n        session: Session = Depends(get_session),\n    ):\n        queue_statuses = {"pending_review", "under_review", "needs_more_evidence"}\n        if review_status is not None and review_status not in queue_statuses:\n            raise HTTPException(status_code=422, detail="Invalid review queue status")\n        statement = select(Correction).where(\n            Correction.status == review_status if review_status else Correction.status.in_(queue_statuses)\n        ).order_by(Correction.created_at, Correction.id)\n        items = []\n        for correction in session.scalars(statement):\n            prediction = require_prediction(correction.prediction_id, session)\n            items.append(ReviewQueueItem(\n                correction_id=correction.id,\n                sample_id=correction.sample_id,\n                prediction_id=correction.prediction_id,\n                field_name=correction.field_name,\n                original_value=correction.original_value_json,\n                proposed_value=correction.proposed_value_json,\n                proposed_asset_id=correction.proposed_asset_id,\n                reason=correction.reason,\n                correction_status=correction.status,\n                prediction_confidence=prediction.confidence,\n                approvals_received=len(approved_reviews(correction.id, session)),\n                approvals_required=approvals_required(correction),\n                created_at=correction.created_at,\n            ))\n        return items\n\n    @app.get("/reviews/{correction_id}", response_model=ReviewDetail)\n    def get_review_detail(correction_id: str, session: Session = Depends(get_session)):\n        return review_detail(require_correction(correction_id, session), session)\n\n    @app.post("/reviews/{correction_id}/decision", response_model=ReviewDecisionResult)\n    def decide_correction(\n        correction_id: str,\n        payload: ReviewDecisionCreate,\n        session: Session = Depends(get_session),\n    ):\n        correction = require_correction(correction_id, session)\n        reviewer_id = payload.reviewer_id.strip()\n        if reviewer_id == correction.submitted_by:\n            raise HTTPException(status_code=403, detail="Submitters cannot review their own corrections")\n        if correction.status in {"approved", "rejected", "withdrawn"}:\n            raise HTTPException(status_code=409, detail=f"A {correction.status} correction cannot be reviewed")\n        duplicate = session.scalar(select(Review).where(\n            Review.correction_id == correction.id,\n            Review.reviewer_id == reviewer_id,\n        ))\n        if duplicate is not None:\n            raise HTTPException(status_code=409, detail="This reviewer already decided this correction")\n\n        notes = (payload.reviewer_notes or "").strip() or None\n        detailed_note_required = correction.field_name in {"coverage_mask", "chlorosis_mask", "growth_form"}\n        if correction.field_name == "coverage_percent":\n            try:\n                change = abs(float(correction.proposed_value_json) - float(correction.original_value_json))\n            except (TypeError, ValueError):\n                change = 0.0\n            detailed_note_required = detailed_note_required or change > 20\n        if payload.decision == "approved" and detailed_note_required and len(notes or "") < 20:\n            raise HTTPException(status_code=422, detail="This correction requires a detailed reviewer note of at least 20 characters")\n        adjusted_value = payload.adjusted_value\n        if adjusted_value is not None:\n            if correction.proposed_asset_id is not None:\n                raise HTTPException(status_code=422, detail="Mask corrections cannot use adjusted_value")\n            if correction.field_name == "coverage_percent":\n                if isinstance(adjusted_value, bool) or not isinstance(adjusted_value, (int, float)) or not 0 <= float(adjusted_value) <= 100:\n                    raise HTTPException(status_code=422, detail="Adjusted coverage must be from 0 to 100")\n                adjusted_value = float(adjusted_value)\n            if correction.field_name == "growth_form" and str(adjusted_value).lower() not in {"crustose", "foliose", "fruticose", "uncertain"}:\n                raise HTTPException(status_code=422, detail="Invalid adjusted growth-form label")\n        if payload.dataset_version_id:\n            assigned = session.get(DatasetVersion, payload.dataset_version_id)\n            if assigned is None or assigned.status != "draft":\n                raise HTTPException(status_code=422, detail="Assigned dataset version must exist and be draft")\n\n        review = Review(\n            correction_id=correction.id,\n            reviewer_id=reviewer_id,\n            decision=payload.decision,\n            reviewer_notes=notes,\n            reviewed_value_json=adjusted_value,\n            assigned_dataset_version_id=payload.dataset_version_id,\n        )\n        session.add(review)\n        session.flush()\n\n        staging: DatasetStagingItem | None = None\n        if payload.decision == "rejected":\n            correction.status = "rejected"\n        elif payload.decision == "needs_more_evidence":\n            correction.status = "needs_more_evidence"\n        else:\n            approvals = approved_reviews(correction.id, session)\n            if len(approvals) >= approvals_required(correction):\n                correction.status = "approved"\n                staging = DatasetStagingItem(\n                    correction_id=correction.id,\n                    sample_id=correction.sample_id,\n                    prediction_id=correction.prediction_id,\n                    field_name=correction.field_name,\n                    verified_value_json=(\n                        {"asset_id": correction.proposed_asset_id}\n                        if correction.proposed_asset_id else (\n                            next((item.reviewed_value_json for item in reversed(approvals) if item.reviewed_value_json is not None), correction.proposed_value_json)\n                        )\n                    ),\n                    annotation_asset_id=correction.proposed_asset_id,\n                    annotation_source="approved_user_correction",\n                    approved_by_json=[item.reviewer_id for item in approvals],\n                    status="ready",\n                )\n                session.add(staging)\n            else:\n                correction.status = "under_review"\n\n        append_event(session, "REVIEW_DECISION_RECORDED", "review", review.id,\n            sample_id=correction.sample_id, actor_id=reviewer_id,\n            payload={"correction_id": correction.id, "decision": review.decision, "adjusted_value": adjusted_value,\n                "dataset_version_id": payload.dataset_version_id})\n        if correction.status == "approved":\n            append_event(session, "CORRECTION_APPROVED", "correction", correction.id,\n                sample_id=correction.sample_id, actor_id=reviewer_id,\n                payload={"staging_item_id": staging.id if staging else None})\n        session.commit()\n        session.refresh(review)\n        session.refresh(correction)\n        if staging is not None:\n            session.refresh(staging)\n        approvals_count = len(approved_reviews(correction.id, session))\n        return ReviewDecisionResult(\n            correction=CorrectionRead.model_validate(correction),\n            review=ReviewRead.model_validate(review),\n            staging_item=DatasetStagingRead.model_validate(staging) if staging else None,\n            approvals_received=approvals_count,\n            approvals_required=approvals_required(correction),\n        )\n\n    @app.get("/dataset-staging", response_model=list[DatasetStagingRead])\n    def list_dataset_staging(session: Session = Depends(get_session)):\n        statement = select(DatasetStagingItem).order_by(\n            DatasetStagingItem.staged_at, DatasetStagingItem.id\n        )\n        return list(session.scalars(statement))\n\n    @app.delete("/samples/{sample_id}/predictions/{prediction_id}", status_code=status.HTTP_204_NO_CONTENT, include_in_schema=False)\n    def immutable_predictions_guard(sample_id: str, prediction_id: str, session: Session = Depends(get_session)) -> Response:\n        del sample_id, prediction_id, session\n        raise HTTPException(status_code=405, detail="Predictions are immutable; create a correction or a new prediction.")\n\n    @app.post("/samples/{sample_id}/assets", response_model=AssetRead, status_code=status.HTTP_201_CREATED)\n    async def upload_asset(\n        sample_id: str,\n        asset_type: str = Form(...),\n        created_by: str = Form(...),\n        source_asset_id: str | None = Form(default=None),\n        file: UploadFile = File(...),\n        session: Session = Depends(get_session),\n    ):\n        require_sample(sample_id, session)\n        normalized_type = safe_asset_type(asset_type)\n        creator = created_by.strip()\n        if not creator or len(creator) > 128:\n            raise HTTPException(status_code=422, detail="created_by must be 1-128 characters")\n        if normalized_type == "original_image":\n            if source_asset_id is not None:\n                raise HTTPException(status_code=422, detail="An original image cannot have a source asset")\n        else:\n            if source_asset_id is None:\n                raise HTTPException(status_code=422, detail="Derived assets require source_asset_id")\n            source = require_asset(source_asset_id, session)\n            if source.sample_id != sample_id:\n                raise HTTPException(status_code=422, detail="Source asset must belong to the same sample")\n\n        data = await file.read(MAX_ASSET_BYTES + 1)\n        original_filename = Path(file.filename or "upload").name[:255]\n        await file.close()\n        if not data:\n            raise HTTPException(status_code=422, detail="Uploaded image is empty")\n        if len(data) > MAX_ASSET_BYTES:\n            raise HTTPException(status_code=413, detail="Image exceeds the 20 MiB limit")\n\n        mime_type, extension, width, height = inspect_image(data)\n        checksum = sha256(data).hexdigest()\n        duplicate = session.scalar(select(SampleAsset).where(\n            SampleAsset.sample_id == sample_id,\n            SampleAsset.checksum == checksum,\n        ))\n        if duplicate is not None:\n            raise HTTPException(status_code=409, detail={\n                "message": "Duplicate image for this sample",\n                "existing_asset_id": duplicate.id,\n            })\n\n        asset_id = new_uuid()\n        relative_path = Path("samples") / sample_id / f"{asset_id}_{normalized_type}{extension}"\n        destination = (asset_root / relative_path).resolve()\n        if asset_root not in destination.parents:\n            raise HTTPException(status_code=500, detail="Invalid asset storage path")\n\n        temporary_path: Path | None = None\n        try:\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            with NamedTemporaryFile(dir=destination.parent, delete=False) as temporary:\n                temporary.write(data)\n                temporary.flush()\n                os.fsync(temporary.fileno())\n                temporary_path = Path(temporary.name)\n            os.replace(temporary_path, destination)\n        except OSError as exc:\n            if temporary_path is not None:\n                temporary_path.unlink(missing_ok=True)\n            raise HTTPException(status_code=500, detail="Asset storage write failed") from exc\n\n        asset = SampleAsset(\n            id=asset_id,\n            sample_id=sample_id,\n            asset_type=normalized_type,\n            storage_path=relative_path.as_posix(),\n            mime_type=mime_type,\n            checksum=checksum,\n            width=width,\n            height=height,\n            created_by=creator,\n            source_asset_id=source_asset_id,\n            original_filename=original_filename,\n        )\n        session.add(asset)\n        session.flush()\n        append_event(session, "IMAGE_UPLOADED", "asset", asset.id, sample_id=sample_id,\n            actor_id=creator, payload={"asset_type": asset.asset_type, "checksum": asset.checksum, "source_asset_id": source_asset_id})\n        try:\n            session.commit()\n        except IntegrityError as exc:\n            session.rollback()\n            destination.unlink(missing_ok=True)\n            duplicate = session.scalar(select(SampleAsset).where(\n                SampleAsset.sample_id == sample_id,\n                SampleAsset.checksum == checksum,\n            ))\n            if duplicate is not None:\n                raise HTTPException(status_code=409, detail={\n                    "message": "Duplicate image for this sample",\n                    "existing_asset_id": duplicate.id,\n                }) from exc\n            raise HTTPException(status_code=500, detail="Asset metadata could not be saved") from exc\n        session.refresh(asset)\n        return asset\n\n    @app.get("/samples/{sample_id}/assets", response_model=list[AssetRead])\n    def list_assets(sample_id: str, session: Session = Depends(get_session)):\n        require_sample(sample_id, session)\n        statement = select(SampleAsset).where(SampleAsset.sample_id == sample_id).order_by(SampleAsset.created_at, SampleAsset.id)\n        return list(session.scalars(statement))\n\n    @app.get("/assets/{asset_id}", response_model=AssetRead)\n    def get_asset(asset_id: str, session: Session = Depends(get_session)):\n        return require_asset(asset_id, session)\n\n    @app.get("/assets/{asset_id}/content", response_class=FileResponse)\n    def get_asset_content(asset_id: str, session: Session = Depends(get_session)):\n        asset = require_asset(asset_id, session)\n        path = (asset_root / asset.storage_path).resolve()\n        if asset_root not in path.parents or not path.is_file():\n            raise HTTPException(status_code=404, detail="Stored asset file not found")\n        return FileResponse(path, media_type=asset.mime_type, filename=asset.original_filename)\n\n    @app.post(\n        "/samples/{sample_id}/analyses/coverage",\n        response_model=PredictionRead,\n        status_code=status.HTTP_201_CREATED,\n    )\n    def analyze_coverage(\n        sample_id: str,\n        payload: CoverageAnalysisRequest,\n        session: Session = Depends(get_session),\n    ):\n        """Run the configured adapter while keeping the API and DB model stable."""\n\n        require_sample(sample_id, session)\n        source = require_asset(payload.image_asset_id, session)\n        if source.sample_id != sample_id:\n            raise HTTPException(status_code=422, detail="Image asset must belong to the sample")\n        if source.asset_type != "original_image":\n            raise HTTPException(status_code=422, detail="Coverage analysis requires an original_image asset")\n        source_path = (asset_root / source.storage_path).resolve()\n        if asset_root not in source_path.parents or not source_path.is_file():\n            raise HTTPException(status_code=404, detail="Stored source image not found")\n\n        try:\n            result = selected_coverage_model.predict(str(source_path))\n        except Exception as exc:\n            raise HTTPException(status_code=503, detail="Coverage model failed") from exc\n\n        required = {"coverage_percent", "confidence", "mask_bytes", "model_version", "is_placeholder"}\n        if not required <= result.keys():\n            raise HTTPException(status_code=500, detail="Coverage adapter returned an invalid result structure")\n        if not 0 <= float(result["coverage_percent"]) <= 100:\n            raise HTTPException(status_code=500, detail="Coverage adapter returned an invalid percentage")\n        if not 0 <= float(result["confidence"]) <= 1:\n            raise HTTPException(status_code=500, detail="Coverage adapter returned invalid confidence")\n        if result.get("is_placeholder") is True and "placeholder" not in str(result["model_version"]).lower():\n            raise HTTPException(status_code=500, detail="Placeholder model version must be explicit")\n\n        mask_data = result["mask_bytes"]\n        if not isinstance(mask_data, bytes):\n            raise HTTPException(status_code=500, detail="Coverage adapter mask must be bytes")\n        mime_type, extension, width, height = inspect_image(mask_data)\n        checksum = sha256(mask_data).hexdigest()\n        mask_asset = session.scalar(select(SampleAsset).where(\n            SampleAsset.sample_id == sample_id,\n            SampleAsset.checksum == checksum,\n        ))\n        if mask_asset is None:\n            asset_id = new_uuid()\n            relative_path = Path("samples") / sample_id / f"{asset_id}_coverage_mask{extension}"\n            destination = (asset_root / relative_path).resolve()\n            try:\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                with NamedTemporaryFile(dir=destination.parent, delete=False) as temporary:\n                    temporary.write(mask_data)\n                    temporary.flush()\n                    os.fsync(temporary.fileno())\n                    temporary_path = Path(temporary.name)\n                os.replace(temporary_path, destination)\n            except OSError as exc:\n                if "temporary_path" in locals():\n                    temporary_path.unlink(missing_ok=True)\n                raise HTTPException(status_code=500, detail="Generated mask storage failed") from exc\n            mask_asset = SampleAsset(\n                id=asset_id,\n                sample_id=sample_id,\n                asset_type="coverage_mask",\n                storage_path=relative_path.as_posix(),\n                mime_type=mime_type,\n                checksum=checksum,\n                width=width,\n                height=height,\n                created_by=str(result["model_version"]),\n                source_asset_id=source.id,\n                original_filename=f"{result[\'model_version\']}_mask.png",\n            )\n            session.add(mask_asset)\n\n        predicted_value = {\n            "coverage_percent": float(result["coverage_percent"]),\n            "mask_asset_id": mask_asset.id,\n            "source_asset_id": source.id,\n            "is_placeholder": bool(result["is_placeholder"]),\n        }\n        prediction = Prediction(\n            sample_id=sample_id,\n            model_type="coverage",\n            model_version=str(result["model_version"]),\n            predicted_value_json=predicted_value,\n            confidence=float(result["confidence"]),\n            status="completed",\n            warnings_json=list(result.get("warnings", [])),\n            is_placeholder=bool(result["is_placeholder"]),\n        )\n        ensure_model_entry(prediction.model_type, prediction.model_version, prediction.is_placeholder, session)\n        session.add(prediction)\n        session.flush()\n        append_event(session, "PREDICTION_CREATED", "prediction", prediction.id,\n            sample_id=sample_id, actor_id=payload.requested_by,\n            payload={"model_type": "coverage", "model_version": prediction.model_version, "source_asset_id": source.id})\n        session.commit()\n        session.refresh(prediction)\n        return prediction\n\n    @app.delete("/assets/{asset_id}", include_in_schema=False)\n    def immutable_assets_guard(asset_id: str):\n        del asset_id\n        raise HTTPException(status_code=405, detail="Assets are immutable; upload a new version to preserve history.")\n\n    register_remaining_routes(app, session_factory, asset_root, selected_adapters, quality_service)\n\n    return app\n\n\napp = create_app()\n', 'tests/test_feature_01.py': 'from pathlib import Path\n\nimport pytest\nfrom fastapi.testclient import TestClient\n\nfrom app.main import create_app\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    database_url = f"sqlite:///{tmp_path / \'feature_01.db\'}"\n    app = create_app(database_url)\n    with TestClient(app) as test_client:\n        yield test_client\n\n\ndef sample_payload() -> dict:\n    return {\n        "study_area_id": "study-cary-01",\n        "user_id": "demo-user",\n        "captured_at": "2026-07-31T12:00:00-04:00",\n        "latitude": 35.7915,\n        "longitude": -78.7811,\n        "substrate_type": "tree_bark",\n        "site_type": "roadside",\n        "notes": "Oak tree approximately 20 meters from road",\n        "status": "submitted",\n    }\n\n\ndef create_sample(client: TestClient) -> dict:\n    response = client.post("/samples", json=sample_payload())\n    assert response.status_code == 201, response.text\n    return response.json()\n\n\ndef test_sample_can_be_created_and_read_without_running_model(client: TestClient):\n    sample = create_sample(client)\n    fetched = client.get(f"/samples/{sample[\'sample_id\']}")\n    predictions = client.get(f"/samples/{sample[\'sample_id\']}/predictions")\n\n    assert fetched.status_code == 200\n    assert fetched.json()["notes"].startswith("Oak tree")\n    assert predictions.status_code == 200\n    assert predictions.json() == []\n\n\ndef test_sample_patch_persists(client: TestClient):\n    sample = create_sample(client)\n    response = client.patch(\n        f"/samples/{sample[\'sample_id\']}",\n        json={"site_type": "school", "notes": "Updated after field verification"},\n    )\n\n    assert response.status_code == 200\n    assert response.json()["site_type"] == "school"\n    assert response.json()["notes"] == "Updated after field verification"\n\n\ndef test_placeholder_and_multiple_model_outputs_can_coexist(client: TestClient):\n    sample = create_sample(client)\n    sample_id = sample["sample_id"]\n    coverage = {\n        "model_type": "coverage",\n        "model_version": "coverage_placeholder_v0",\n        "predicted_value": {"coverage_percent": 37.4},\n        "confidence": 0.72,\n        "status": "completed",\n        "warnings": ["COVERAGE_MODEL_PLACEHOLDER"],\n        "is_placeholder": True,\n    }\n    growth_form = {\n        "model_type": "growth_form",\n        "model_version": "growthform_v1",\n        "predicted_value": {\n            "class": "foliose",\n            "class_probabilities": {\n                "crustose": 0.12,\n                "foliose": 0.74,\n                "fruticose": 0.09,\n                "uncertain": 0.05,\n            },\n        },\n        "confidence": 0.74,\n        "status": "completed",\n        "warnings": [],\n        "is_placeholder": False,\n    }\n\n    assert client.post(f"/samples/{sample_id}/predictions", json=coverage).status_code == 201\n    assert client.post(f"/samples/{sample_id}/predictions", json=growth_form).status_code == 201\n    results = client.get(f"/samples/{sample_id}/predictions").json()\n\n    assert len(results) == 2\n    assert {result["model_type"] for result in results} == {"coverage", "growth_form"}\n    assert next(r for r in results if r["model_type"] == "coverage")["is_placeholder"] is True\n    assert next(r for r in results if r["model_type"] == "growth_form")["predicted_value"]["class"] == "foliose"\n\n\ndef test_placeholder_label_is_required(client: TestClient):\n    sample = create_sample(client)\n    response = client.post(\n        f"/samples/{sample[\'sample_id\']}/predictions",\n        json={\n            "model_type": "coverage",\n            "model_version": "coverage_placeholder_v0",\n            "predicted_value": {"coverage_percent": 30.0},\n            "confidence": 0.5,\n            "is_placeholder": False,\n        },\n    )\n    assert response.status_code == 422\n\n\ndef test_coordinates_are_validated_as_pair(client: TestClient):\n    payload = sample_payload()\n    payload.pop("longitude")\n    response = client.post("/samples", json=payload)\n    assert response.status_code == 422\n\n\ndef test_missing_sample_returns_404(client: TestClient):\n    response = client.get("/samples/not-a-real-id")\n    assert response.status_code == 404\n\n', 'tests/test_feature_02.py': 'from io import BytesIO\nfrom pathlib import Path\nimport sqlite3\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.main import MAX_ASSET_BYTES, create_app\n\n\ndef image_bytes(image_format: str = "PNG", color=(20, 140, 70), size=(12, 8)) -> bytes:\n    buffer = BytesIO()\n    Image.new("RGB", size, color).save(buffer, format=image_format)\n    return buffer.getvalue()\n\n\ndef sample_payload() -> dict:\n    return {\n        "study_area_id": "study-cary-01",\n        "user_id": "demo-user",\n        "captured_at": "2026-07-31T12:00:00-04:00",\n        "latitude": 35.7915,\n        "longitude": -78.7811,\n        "substrate_type": "tree_bark",\n        "site_type": "roadside",\n        "notes": "Feature 2 test",\n        "status": "submitted",\n    }\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    app = create_app(\n        f"sqlite:///{tmp_path / \'feature_02.db\'}",\n        storage_root=tmp_path / "assets",\n    )\n    with TestClient(app) as test_client:\n        yield test_client\n\n\ndef create_sample(client: TestClient) -> str:\n    response = client.post("/samples", json=sample_payload())\n    assert response.status_code == 201, response.text\n    return response.json()["sample_id"]\n\n\ndef upload(\n    client: TestClient,\n    sample_id: str,\n    data: bytes,\n    asset_type: str = "original_image",\n    source_asset_id: str | None = None,\n    filename: str = "lichen.png",\n):\n    form = {"asset_type": asset_type, "created_by": "test-suite"}\n    if source_asset_id is not None:\n        form["source_asset_id"] = source_asset_id\n    return client.post(\n        f"/samples/{sample_id}/assets",\n        data=form,\n        files={"file": (filename, data, "application/octet-stream")},\n    )\n\n\ndef test_original_upload_creates_metadata_and_retrievable_file(client: TestClient):\n    sample_id = create_sample(client)\n    data = image_bytes(size=(17, 11))\n    response = upload(client, sample_id, data)\n\n    assert response.status_code == 201, response.text\n    asset = response.json()\n    assert asset["asset_type"] == "original_image"\n    assert asset["mime_type"] == "image/png"\n    assert (asset["width"], asset["height"]) == (17, 11)\n    assert len(asset["checksum"]) == 64\n    assert asset["source_asset_id"] is None\n    assert client.get(f"/assets/{asset[\'asset_id\']}").json() == asset\n    content = client.get(f"/assets/{asset[\'asset_id\']}/content")\n    assert content.status_code == 200\n    assert content.content == data\n\n\ndef test_same_file_is_detected_as_duplicate_for_one_sample(client: TestClient):\n    sample_id = create_sample(client)\n    data = image_bytes()\n    first = upload(client, sample_id, data)\n    duplicate = upload(client, sample_id, data)\n\n    assert first.status_code == 201\n    assert duplicate.status_code == 409\n    assert duplicate.json()["detail"]["existing_asset_id"] == first.json()["asset_id"]\n    assert len(client.get(f"/samples/{sample_id}/assets").json()) == 1\n\n\ndef test_same_file_is_allowed_for_a_different_sample(client: TestClient):\n    first_sample = create_sample(client)\n    second_sample = create_sample(client)\n    data = image_bytes()\n    assert upload(client, first_sample, data).status_code == 201\n    assert upload(client, second_sample, data).status_code == 201\n\n\ndef test_derived_asset_links_to_original_and_history_is_preserved(client: TestClient):\n    sample_id = create_sample(client)\n    original = upload(client, sample_id, image_bytes(color=(10, 20, 30))).json()\n    overlay_1 = upload(\n        client,\n        sample_id,\n        image_bytes(color=(200, 10, 20)),\n        asset_type="coverage_overlay",\n        source_asset_id=original["asset_id"],\n        filename="overlay-v0.png",\n    )\n    overlay_2 = upload(\n        client,\n        sample_id,\n        image_bytes(color=(210, 20, 30)),\n        asset_type="coverage_overlay",\n        source_asset_id=original["asset_id"],\n        filename="overlay-v1.png",\n    )\n\n    assert overlay_1.status_code == 201\n    assert overlay_2.status_code == 201\n    assert overlay_1.json()["source_asset_id"] == original["asset_id"]\n    assets = client.get(f"/samples/{sample_id}/assets").json()\n    assert len(assets) == 3\n    assert {item["asset_id"] for item in assets} == {\n        original["asset_id"], overlay_1.json()["asset_id"], overlay_2.json()["asset_id"]\n    }\n    assert client.delete(f"/assets/{overlay_1.json()[\'asset_id\']}").status_code == 405\n\n\ndef test_derived_asset_requires_a_source(client: TestClient):\n    sample_id = create_sample(client)\n    response = upload(client, sample_id, image_bytes(), asset_type="coverage_mask")\n    assert response.status_code == 422\n    assert "source_asset_id" in response.text\n\n\ndef test_source_asset_must_exist_and_belong_to_same_sample(client: TestClient):\n    first_sample = create_sample(client)\n    second_sample = create_sample(client)\n    original = upload(client, first_sample, image_bytes()).json()\n    missing = upload(\n        client, second_sample, image_bytes(color=(1, 2, 3)),\n        asset_type="coverage_mask", source_asset_id="missing-id"\n    )\n    wrong_sample = upload(\n        client, second_sample, image_bytes(color=(4, 5, 6)),\n        asset_type="coverage_mask", source_asset_id=original["asset_id"]\n    )\n    assert missing.status_code == 404\n    assert wrong_sample.status_code == 422\n\n\ndef test_corrupted_and_unsupported_images_are_rejected(client: TestClient):\n    sample_id = create_sample(client)\n    corrupt = upload(client, sample_id, b"not an image")\n    unsupported = upload(client, sample_id, image_bytes("GIF"), filename="lichen.gif")\n    assert corrupt.status_code == 422\n    assert unsupported.status_code == 415\n    assert client.get(f"/samples/{sample_id}/assets").json() == []\n\n\ndef test_oversized_file_is_rejected_before_image_parsing(client: TestClient):\n    sample_id = create_sample(client)\n    response = upload(client, sample_id, b"x" * (MAX_ASSET_BYTES + 1))\n    assert response.status_code == 413\n\n\ndef test_missing_sample_and_asset_return_404(client: TestClient):\n    response = upload(client, "missing-sample", image_bytes())\n    assert response.status_code == 404\n    assert client.get("/assets/missing-asset").status_code == 404\n\n\ndef test_existing_feature_1_database_is_migrated(tmp_path: Path):\n    database_path = tmp_path / "old_feature_1.db"\n    with sqlite3.connect(database_path) as connection:\n        connection.execute("""\n            CREATE TABLE sample_assets (\n                id VARCHAR(36) PRIMARY KEY,\n                sample_id VARCHAR(36) NOT NULL,\n                asset_type VARCHAR(64) NOT NULL,\n                storage_path TEXT NOT NULL,\n                mime_type VARCHAR(128) NOT NULL,\n                checksum VARCHAR(128) NOT NULL,\n                created_at DATETIME NOT NULL\n            )\n        """)\n    app = create_app(f"sqlite:///{database_path}", storage_root=tmp_path / "assets")\n    with TestClient(app):\n        pass\n    with sqlite3.connect(database_path) as connection:\n        columns = {row[1] for row in connection.execute("PRAGMA table_info(sample_assets)")}\n    assert {"width", "height", "created_by", "source_asset_id", "original_filename"} <= columns\n', 'tests/test_feature_03.py': 'from io import BytesIO\nfrom pathlib import Path\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.inference import CoverageModel\nfrom app.main import create_app\n\n\ndef png_bytes(color=(40, 120, 65), size=(20, 14)) -> bytes:\n    buffer = BytesIO()\n    Image.new("RGB", size, color).save(buffer, format="PNG")\n    return buffer.getvalue()\n\n\ndef sample_payload() -> dict:\n    return {\n        "captured_at": "2026-07-31T12:00:00-04:00",\n        "latitude": 35.7915,\n        "longitude": -78.7811,\n        "status": "submitted",\n    }\n\n\ndef create_sample_and_image(client: TestClient, color=(40, 120, 65)) -> tuple[str, str]:\n    sample = client.post("/samples", json=sample_payload()).json()\n    upload = client.post(\n        f"/samples/{sample[\'sample_id\']}/assets",\n        data={"asset_type": "original_image", "created_by": "feature-3-test"},\n        files={"file": ("lichen.png", png_bytes(color), "image/png")},\n    )\n    assert upload.status_code == 201, upload.text\n    return sample["sample_id"], upload.json()["asset_id"]\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    app = create_app(\n        f"sqlite:///{tmp_path / \'feature_03.db\'}",\n        storage_root=tmp_path / "assets",\n    )\n    with TestClient(app) as test_client:\n        yield test_client\n\n\ndef test_placeholder_completes_analysis_with_explicit_label(client: TestClient):\n    sample_id, image_id = create_sample_and_image(client)\n    response = client.post(\n        f"/samples/{sample_id}/analyses/coverage",\n        json={"image_asset_id": image_id, "requested_by": "tester"},\n    )\n    assert response.status_code == 201, response.text\n    prediction = response.json()\n    assert prediction["model_type"] == "coverage"\n    assert prediction["model_version"] == "coverage_placeholder_v0"\n    assert prediction["is_placeholder"] is True\n    assert prediction["predicted_value"]["is_placeholder"] is True\n    assert 25 <= prediction["predicted_value"]["coverage_percent"] <= 60\n    assert prediction["confidence"] == 0.70\n    assert "DEVELOPMENT_PLACEHOLDER" in prediction["warnings"][0]\n\n\ndef test_placeholder_creates_retrievable_mask_with_source_lineage(client: TestClient):\n    sample_id, image_id = create_sample_and_image(client)\n    prediction = client.post(\n        f"/samples/{sample_id}/analyses/coverage",\n        json={"image_asset_id": image_id},\n    ).json()\n    mask_id = prediction["predicted_value"]["mask_asset_id"]\n    mask = client.get(f"/assets/{mask_id}")\n    assert mask.status_code == 200\n    assert mask.json()["asset_type"] == "coverage_mask"\n    assert mask.json()["source_asset_id"] == image_id\n    assert client.get(f"/assets/{mask_id}/content").status_code == 200\n\n\ndef test_repeated_placeholder_run_is_deterministic_and_reuses_mask(client: TestClient):\n    sample_id, image_id = create_sample_and_image(client)\n    first = client.post(\n        f"/samples/{sample_id}/analyses/coverage", json={"image_asset_id": image_id}\n    ).json()\n    second = client.post(\n        f"/samples/{sample_id}/analyses/coverage", json={"image_asset_id": image_id}\n    ).json()\n    assert first["predicted_value"] == second["predicted_value"]\n    assert first["prediction_id"] != second["prediction_id"]\n    assert len(client.get(f"/samples/{sample_id}/assets").json()) == 2\n\n\nclass FinalAdapter(CoverageModel):\n    model_version = "coverage_unet_candidate_v1"\n    is_placeholder = False\n\n    def predict(self, image_path: str) -> dict:\n        with Image.open(image_path) as image:\n            mask = Image.new("L", image.size, 255)\n        buffer = BytesIO()\n        mask.save(buffer, format="PNG")\n        return {\n            "coverage_percent": 83.2,\n            "confidence": 0.91,\n            "mask_bytes": buffer.getvalue(),\n            "mask_mime_type": "image/png",\n            "model_version": self.model_version,\n            "is_placeholder": False,\n            "warnings": [],\n        }\n\n\ndef test_final_adapter_uses_identical_endpoint_and_database_schema(tmp_path: Path):\n    app = create_app(\n        f"sqlite:///{tmp_path / \'final_adapter.db\'}",\n        storage_root=tmp_path / "assets",\n        coverage_model=FinalAdapter(),\n    )\n    with TestClient(app) as client:\n        sample_id, image_id = create_sample_and_image(client)\n        response = client.post(\n            f"/samples/{sample_id}/analyses/coverage", json={"image_asset_id": image_id}\n        )\n        assert response.status_code == 201\n        result = response.json()\n        assert result["model_version"] == "coverage_unet_candidate_v1"\n        assert result["is_placeholder"] is False\n        assert result["predicted_value"]["coverage_percent"] == 83.2\n\n\ndef test_analysis_rejects_missing_cross_sample_and_derived_assets(client: TestClient):\n    first_sample, first_image = create_sample_and_image(client)\n    second_sample, second_image = create_sample_and_image(client, color=(90, 40, 20))\n    missing = client.post(\n        f"/samples/{first_sample}/analyses/coverage", json={"image_asset_id": "missing"}\n    )\n    wrong_sample = client.post(\n        f"/samples/{first_sample}/analyses/coverage", json={"image_asset_id": second_image}\n    )\n    overlay = client.post(\n        f"/samples/{first_sample}/assets",\n        data={\n            "asset_type": "coverage_overlay",\n            "created_by": "test",\n            "source_asset_id": first_image,\n        },\n        files={"file": ("overlay.png", png_bytes((200, 10, 10)), "image/png")},\n    ).json()\n    derived = client.post(\n        f"/samples/{first_sample}/analyses/coverage",\n        json={"image_asset_id": overlay["asset_id"]},\n    )\n    assert missing.status_code == 404\n    assert wrong_sample.status_code == 422\n    assert derived.status_code == 422\n\n\nclass BrokenAdapter(CoverageModel):\n    model_version = "broken_v1"\n    is_placeholder = False\n\n    def predict(self, image_path: str) -> dict:\n        return {"coverage_percent": 40.0}\n\n\ndef test_invalid_adapter_result_is_not_persisted(tmp_path: Path):\n    app = create_app(\n        f"sqlite:///{tmp_path / \'broken.db\'}",\n        storage_root=tmp_path / "assets",\n        coverage_model=BrokenAdapter(),\n    )\n    with TestClient(app) as client:\n        sample_id, image_id = create_sample_and_image(client)\n        response = client.post(\n            f"/samples/{sample_id}/analyses/coverage", json={"image_asset_id": image_id}\n        )\n        assert response.status_code == 500\n        assert client.get(f"/samples/{sample_id}/predictions").json() == []\n', 'tests/test_feature_04.py': 'from io import BytesIO\nfrom pathlib import Path\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.main import create_app\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    app = create_app(f"sqlite:///{tmp_path / \'feature_04.db\'}", storage_root=tmp_path / "assets")\n    with TestClient(app) as test_client:\n        yield test_client\n\n\ndef create_sample(client: TestClient) -> str:\n    response = client.post("/samples", json={\n        "captured_at": "2026-07-31T12:00:00-04:00",\n        "latitude": 35.7915,\n        "longitude": -78.7811,\n        "substrate_type": "tree_bark",\n        "site_type": "roadside",\n        "notes": "original note",\n        "status": "submitted",\n    })\n    assert response.status_code == 201\n    return response.json()["sample_id"]\n\n\ndef attach_prediction(client: TestClient, sample_id: str, model_type="coverage", value=None) -> dict:\n    if value is None:\n        value = {"coverage_percent": 42.1, "mask_asset_id": None}\n    response = client.post(f"/samples/{sample_id}/predictions", json={\n        "model_type": model_type,\n        "model_version": f"{model_type}_candidate_v1",\n        "predicted_value": value,\n        "confidence": 0.78,\n        "status": "completed",\n        "warnings": [],\n        "is_placeholder": False,\n    })\n    assert response.status_code == 201, response.text\n    return response.json()\n\n\ndef correction_payload(**overrides) -> dict:\n    payload = {\n        "field_name": "coverage_percent",\n        "proposed_value": 35.0,\n        "correction_method": "manual_estimate",\n        "reason": "Manual grid estimate shows a smaller visible lichen area.",\n        "submitted_by": "citizen-1",\n    }\n    payload.update(overrides)\n    return payload\n\n\ndef png_bytes(color=(40, 150, 70)) -> bytes:\n    buffer = BytesIO()\n    Image.new("L", (10, 8), color[0]).save(buffer, format="PNG")\n    return buffer.getvalue()\n\n\ndef test_submit_value_correction_preserves_original_prediction(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    response = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(),\n    )\n    assert response.status_code == 201, response.text\n    correction = response.json()\n    assert correction["original_value"] == 42.1\n    assert correction["proposed_value"] == 35.0\n    assert correction["status"] == "pending_review"\n    unchanged = client.get(f"/samples/{sample_id}/predictions").json()[0]\n    assert unchanged == prediction\n\n\ndef test_full_history_and_status_filter_are_retrievable(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    first = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(proposed_value=37.0),\n    ).json()\n    second = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(proposed_value=33.0, reason="A second independent grid count produced a lower estimate."),\n    ).json()\n    history = client.get(f"/corrections?sample_id={sample_id}").json()\n    pending = client.get("/corrections?status=pending_review").json()\n    assert [item["correction_id"] for item in history] == [first["correction_id"], second["correction_id"]]\n    assert len(pending) == 2\n    assert client.get(f"/corrections/{first[\'correction_id\']}").json() == first\n\n\ndef test_withdraw_requires_submitter_and_keeps_audit_record(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    correction = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections", json=correction_payload()\n    ).json()\n    denied = client.patch(\n        f"/corrections/{correction[\'correction_id\']}/withdraw", json={"requested_by": "someone-else"}\n    )\n    withdrawn = client.patch(\n        f"/corrections/{correction[\'correction_id\']}/withdraw", json={"requested_by": "citizen-1"}\n    )\n    assert denied.status_code == 403\n    assert withdrawn.status_code == 200\n    assert withdrawn.json()["status"] == "withdrawn"\n    assert withdrawn.json()["withdrawn_at"] is not None\n    assert client.delete(f"/corrections/{correction[\'correction_id\']}").status_code == 405\n\n\ndef test_growth_form_correction_is_validated(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id, "growth_form", {"class": "crustose", "class_probabilities": {"crustose": .8}})\n    valid = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="growth_form", proposed_value="foliose"),\n    )\n    invalid = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="growth_form", proposed_value="species_x"),\n    )\n    assert valid.status_code == 201\n    assert valid.json()["original_value"] == "crustose"\n    assert invalid.status_code == 422\n\n\ndef test_mask_correction_references_reviewed_asset(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    original = client.post(\n        f"/samples/{sample_id}/assets",\n        data={"asset_type": "original_image", "created_by": "tester"},\n        files={"file": ("original.png", png_bytes((30, 0, 0)), "image/png")},\n    ).json()\n    reviewed = client.post(\n        f"/samples/{sample_id}/assets",\n        data={"asset_type": "reviewed_mask", "created_by": "citizen-1", "source_asset_id": original["asset_id"]},\n        files={"file": ("reviewed.png", png_bytes((220, 0, 0)), "image/png")},\n    ).json()\n    response = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="coverage_mask", proposed_value=None, proposed_asset_id=reviewed["asset_id"]),\n    )\n    assert response.status_code == 201, response.text\n    assert response.json()["proposed_asset_id"] == reviewed["asset_id"]\n\n\ndef test_mask_from_another_sample_or_wrong_asset_type_is_rejected(client: TestClient):\n    first = create_sample(client)\n    second = create_sample(client)\n    prediction = attach_prediction(client, first)\n    second_original = client.post(\n        f"/samples/{second}/assets", data={"asset_type": "original_image", "created_by": "tester"},\n        files={"file": ("other.png", png_bytes(), "image/png")},\n    ).json()\n    wrong_sample = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="coverage_mask", proposed_value=None, proposed_asset_id=second_original["asset_id"]),\n    )\n    assert wrong_sample.status_code == 422\n\n\ndef test_only_safe_sample_metadata_is_correctable(client: TestClient):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    allowed = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="sample_metadata.site_type", proposed_value="school"),\n    )\n    blocked = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name="sample_metadata.latitude", proposed_value=36.0),\n    )\n    assert allowed.status_code == 201\n    assert allowed.json()["original_value"] == "roadside"\n    assert blocked.status_code == 422\n\n\n@pytest.mark.parametrize("field", ["environmental_risk", "model_confidence", "geomorphology", "latitude"])\ndef test_derived_and_protected_fields_cannot_be_corrected(client: TestClient, field: str):\n    sample_id = create_sample(client)\n    prediction = attach_prediction(client, sample_id)\n    response = client.post(\n        f"/predictions/{prediction[\'prediction_id\']}/corrections",\n        json=correction_payload(field_name=field),\n    )\n    assert response.status_code == 422\n', 'tests/test_feature_05.py': 'from io import BytesIO\nfrom pathlib import Path\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.main import create_app\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    app = create_app(f"sqlite:///{tmp_path / \'feature_05.db\'}", storage_root=tmp_path / "assets")\n    with TestClient(app) as test_client:\n        yield test_client\n\n\ndef image_bytes(value=80) -> bytes:\n    buffer = BytesIO()\n    Image.new("L", (12, 9), value).save(buffer, format="PNG")\n    return buffer.getvalue()\n\n\ndef make_case(client: TestClient, *, model_type="coverage", predicted=None, field="coverage_percent", proposed=35.0, submitter="citizen-1") -> tuple[str, dict, dict]:\n    sample_response = client.post("/samples", json={\n        "captured_at": "2026-07-31T12:00:00-04:00", "latitude": 35.7915,\n        "longitude": -78.7811, "substrate_type": "tree_bark", "site_type": "roadside",\n        "notes": "review workflow test", "status": "submitted",\n    })\n    sample_id = sample_response.json()["sample_id"]\n    if predicted is None:\n        predicted = {"coverage_percent": 42.1, "mask_asset_id": None}\n    prediction = client.post(f"/samples/{sample_id}/predictions", json={\n        "model_type": model_type, "model_version": f"{model_type}_candidate_v1",\n        "predicted_value": predicted, "confidence": .78, "status": "completed",\n        "warnings": [], "is_placeholder": False,\n    }).json()\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": field, "proposed_value": proposed, "correction_method": "manual_estimate",\n        "reason": "Visible evidence supports the proposed corrected observation.", "submitted_by": submitter,\n    })\n    assert correction.status_code == 201, correction.text\n    return sample_id, prediction, correction.json()\n\n\ndef approve(client: TestClient, correction_id: str, reviewer="reviewer-1", notes=None):\n    return client.post(f"/reviews/{correction_id}/decision", json={\n        "reviewer_id": reviewer, "decision": "approved", "reviewer_notes": notes,\n    })\n\n\ndef test_review_queue_contains_priority_context(client: TestClient):\n    _, prediction, correction = make_case(client)\n    queue = client.get("/reviews/queue").json()\n    assert len(queue) == 1\n    item = queue[0]\n    assert item["correction_id"] == correction["correction_id"]\n    assert item["prediction_confidence"] == prediction["confidence"]\n    assert item["original_value"] == 42.1\n    assert item["proposed_value"] == 35.0\n    assert item["approvals_required"] == 1\n\n\ndef test_review_detail_includes_evidence_metadata_and_history(client: TestClient):\n    sample_id, prediction, first = make_case(client)\n    original = client.post(\n        f"/samples/{sample_id}/assets", data={"asset_type": "original_image", "created_by": "tester"},\n        files={"file": ("original.png", image_bytes(), "image/png")},\n    )\n    assert original.status_code == 201\n    second = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": "coverage_percent", "proposed_value": 36.0, "correction_method": "manual_grid",\n        "reason": "A second grid count supports another proposed estimate.", "submitted_by": "citizen-2",\n    }).json()\n    detail = client.get(f"/reviews/{second[\'correction_id\']}").json()\n    assert detail["sample"]["site_type"] == "roadside"\n    assert detail["prediction"]["confidence"] == .78\n    assert detail["assets"][0]["asset_type"] == "original_image"\n    assert detail["previous_corrections"][0]["correction_id"] == first["correction_id"]\n    assert detail["correction"]["reason"]\n\n\ndef test_one_reviewer_approval_stages_value_correction(client: TestClient):\n    _, _, correction = make_case(client)\n    response = approve(client, correction["correction_id"])\n    assert response.status_code == 200, response.text\n    result = response.json()\n    assert result["correction"]["status"] == "approved"\n    assert result["review"]["reviewer_id"] == "reviewer-1"\n    assert result["review"]["reviewed_at"] is not None\n    assert result["staging_item"]["verified_value"] == 35.0\n    assert result["staging_item"]["annotation_source"] == "approved_user_correction"\n    assert client.get("/reviews/queue").json() == []\n    assert len(client.get("/dataset-staging").json()) == 1\n\n\ndef test_rejection_is_audited_and_never_staged(client: TestClient):\n    _, _, correction = make_case(client)\n    response = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n        "reviewer_id": "reviewer-1", "decision": "rejected",\n        "reviewer_notes": "The proposed value is not supported by the visible boundaries.",\n    })\n    assert response.status_code == 200\n    assert response.json()["correction"]["status"] == "rejected"\n    assert response.json()["staging_item"] is None\n    detail = client.get(f"/reviews/{correction[\'correction_id\']}").json()\n    assert detail["reviews"][0]["decision"] == "rejected"\n    assert client.get("/dataset-staging").json() == []\n\n\ndef test_needs_more_evidence_remains_in_queue(client: TestClient):\n    _, _, correction = make_case(client)\n    missing_note = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n        "reviewer_id": "reviewer-1", "decision": "needs_more_evidence",\n    })\n    response = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n        "reviewer_id": "reviewer-1", "decision": "needs_more_evidence",\n        "reviewer_notes": "Please provide a clearer image with the full lichen boundary.",\n    })\n    assert missing_note.status_code == 422\n    assert response.status_code == 200\n    assert response.json()["correction"]["status"] == "needs_more_evidence"\n    assert client.get("/reviews/queue?status=needs_more_evidence").json()[0]["correction_id"] == correction["correction_id"]\n\n\ndef test_submitter_cannot_review_and_reviewer_cannot_vote_twice(client: TestClient):\n    sample_id, prediction, value_correction = make_case(client)\n    own = approve(client, value_correction["correction_id"], reviewer="citizen-1")\n    assert own.status_code == 403\n\n    original = client.post(\n        f"/samples/{sample_id}/assets", data={"asset_type": "original_image", "created_by": "tester"},\n        files={"file": ("original.png", image_bytes(20), "image/png")},\n    ).json()\n    reviewed = client.post(\n        f"/samples/{sample_id}/assets", data={"asset_type": "reviewed_mask", "created_by": "citizen-1", "source_asset_id": original["asset_id"]},\n        files={"file": ("reviewed.png", image_bytes(220), "image/png")},\n    ).json()\n    mask_correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": "coverage_mask", "proposed_value": None, "proposed_asset_id": reviewed["asset_id"],\n        "reason": "The manually traced boundary differs from the generated mask.", "submitted_by": "citizen-1",\n    }).json()\n    first = approve(client, mask_correction["correction_id"], notes="The corrected boundary follows visible lichen edges.")\n    duplicate = approve(client, mask_correction["correction_id"], notes="Attempting a duplicate decision with the same reviewer.")\n    assert first.status_code == 200\n    assert duplicate.status_code == 409\n\n\ndef test_mask_requires_two_independent_approvals_before_staging(client: TestClient):\n    sample_id, prediction, _ = make_case(client)\n    original = client.post(\n        f"/samples/{sample_id}/assets", data={"asset_type": "original_image", "created_by": "tester"},\n        files={"file": ("original.png", image_bytes(15), "image/png")},\n    ).json()\n    reviewed = client.post(\n        f"/samples/{sample_id}/assets", data={"asset_type": "reviewed_mask", "created_by": "citizen-1", "source_asset_id": original["asset_id"]},\n        files={"file": ("reviewed.png", image_bytes(230), "image/png")},\n    ).json()\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": "coverage_mask", "proposed_value": None, "proposed_asset_id": reviewed["asset_id"],\n        "reason": "Manual tracing follows the visible lichen boundary more closely.", "submitted_by": "citizen-1",\n    }).json()\n    first = approve(client, correction["correction_id"], "reviewer-a", "Visible edge alignment supports this corrected mask.")\n    assert first.json()["correction"]["status"] == "under_review"\n    assert first.json()["staging_item"] is None\n    second = approve(client, correction["correction_id"], "reviewer-b", "Independent inspection confirms the corrected boundary.")\n    assert second.status_code == 200, second.text\n    assert second.json()["correction"]["status"] == "approved"\n    assert second.json()["approvals_received"] == 2\n    assert second.json()["staging_item"]["annotation_asset_id"] == reviewed["asset_id"]\n    assert second.json()["staging_item"]["approved_by"] == ["reviewer-a", "reviewer-b"]\n\n\ndef test_large_coverage_change_requires_detailed_reviewer_note(client: TestClient):\n    _, _, correction = make_case(client, proposed=10.0)\n    detail = client.get(f"/reviews/{correction[\'correction_id\']}").json()\n    assert detail["review_requirements"]["detailed_note_required"] is True\n    assert "LARGE_COVERAGE_CHANGE" in detail["review_requirements"]["warnings"][0]\n    short = approve(client, correction["correction_id"], notes="Looks valid")\n    detailed = approve(client, correction["correction_id"], notes="A manual 100-cell grid count supports the 10 percent estimate.")\n    assert short.status_code == 422\n    assert detailed.status_code == 200\n\n\ndef test_growth_form_approval_requires_written_evidence(client: TestClient):\n    _, _, correction = make_case(\n        client, model_type="growth_form", predicted={"class": "crustose"},\n        field="growth_form", proposed="foliose",\n    )\n    short = approve(client, correction["correction_id"], notes="Agree")\n    detailed = approve(client, correction["correction_id"], notes="Leaf-like lobes visible in the image support foliose morphology.")\n    assert short.status_code == 422\n    assert detailed.status_code == 200\n\n\ndef test_terminal_decisions_cannot_be_overwritten(client: TestClient):\n    _, _, correction = make_case(client)\n    assert approve(client, correction["correction_id"]).status_code == 200\n    second = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n        "reviewer_id": "reviewer-2", "decision": "rejected", "reviewer_notes": "Trying to replace the prior decision.",\n    })\n    withdrawal = client.patch(f"/corrections/{correction[\'correction_id\']}/withdraw", json={"requested_by": "citizen-1"})\n    assert second.status_code == 409\n    assert withdrawal.status_code == 409\n', 'tests/test_features_06_10.py': 'from hashlib import sha256\nfrom io import BytesIO\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.main import create_app\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    app = create_app(\n        f"sqlite:///{tmp_path / \'features_06_10.db\'}",\n        storage_root=tmp_path / "assets",\n        quality_thresholds={"min_width": 8, "min_height": 8, "min_blur_score": 0,\n            "min_brightness": 0, "max_brightness": 1, "max_overexposed_fraction": 1, "min_contrast": 0},\n    )\n    with TestClient(app) as test_client:\n        test_client.storage_root = tmp_path / "assets"\n        yield test_client\n\n\ndef png(value: int, size=(32, 32)) -> bytes:\n    buffer = BytesIO(); Image.new("L", size, value).save(buffer, format="PNG"); return buffer.getvalue()\n\n\ndef sample(client: TestClient, *, area="area-1", site="roadside") -> str:\n    response = client.post("/samples", json={"study_area_id": area, "user_id": "field-user",\n        "captured_at": "2026-07-31T12:00:00-04:00", "latitude": 35.7915,\n        "longitude": -78.7811, "substrate_type": "tree_bark", "site_type": site, "status": "submitted"})\n    assert response.status_code == 201, response.text; return response.json()["sample_id"]\n\n\ndef upload(client: TestClient, sample_id: str, *, kind="original_image", value=80, source=None, size=(32,32)) -> dict:\n    data = {"asset_type": kind, "created_by": "tester"}\n    if source: data["source_asset_id"] = source\n    response = client.post(f"/samples/{sample_id}/assets", data=data,\n        files={"file": (f"{kind}.png", png(value, size), "image/png")})\n    assert response.status_code == 201, response.text; return response.json()\n\n\ndef approved_value_case(client: TestClient, *, area="area-1", site="roadside"):\n    sample_id = sample(client, area=area, site=site); original = upload(client, sample_id)\n    prediction = client.post(f"/samples/{sample_id}/predictions", json={"model_type": "coverage",\n        "model_version": "coverage_candidate_v1", "predicted_value": {"coverage_percent": 42.0},\n        "confidence": .78, "is_placeholder": False}).json()\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": "coverage_percent", "proposed_value": 35.0, "correction_method": "manual_grid",\n        "reason": "A manual grid count supports the corrected coverage value.", "submitted_by": "field-user"}).json()\n    decision = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n        "reviewer_id": "reviewer-1", "decision": "approved", "reviewer_notes": "Independent grid count confirms the value."})\n    assert decision.status_code == 200, decision.text\n    return sample_id, original, decision.json()["staging_item"]\n\n\ndef approved_mask_case(client: TestClient):\n    sample_id = sample(client); original = upload(client, sample_id, value=50)\n    prediction = client.post(f"/samples/{sample_id}/predictions", json={"model_type": "coverage",\n        "model_version": "coverage_candidate_v1", "predicted_value": {"coverage_percent": 42.0},\n        "confidence": .78, "is_placeholder": False}).json()\n    reviewed = upload(client, sample_id, kind="reviewed_mask", value=200, source=original["asset_id"])\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={\n        "field_name": "coverage_mask", "proposed_asset_id": reviewed["asset_id"],\n        "reason": "Manual tracing follows the visible lichen edge more accurately.", "submitted_by": "field-user"}).json()\n    for reviewer in ("reviewer-a", "reviewer-b"):\n        result = client.post(f"/reviews/{correction[\'correction_id\']}/decision", json={\n            "reviewer_id": reviewer, "decision": "approved", "reviewer_notes": "Independent inspection confirms boundary alignment."})\n    assert result.json()["correction"]["status"] == "approved"\n    return sample_id, original, result.json()["staging_item"]\n\n\ndef version(client: TestClient, name="lichen_dataset_v2.1") -> dict:\n    response = client.post("/dataset-versions", json={"name": name, "description": "Reviewed pilot data", "created_by": "dataset-admin"})\n    assert response.status_code == 201, response.text; return response.json()\n\n\ndef add_item(client, version_id, staging, original, split="train", site_group=None):\n    payload = {"staging_item_id": staging["staging_item_id"], "image_asset_id": original["asset_id"], "split": split}\n    if site_group: payload["site_group"] = site_group\n    return client.post(f"/dataset-versions/{version_id}/items", json=payload)\n\n\ndef test_dataset_version_add_manifest_freeze_and_immutability(client):\n    _, original, staging = approved_value_case(client); dataset = version(client)\n    added = add_item(client, dataset["dataset_version_id"], staging, original)\n    assert added.status_code == 201, added.text\n    manifest = client.get(f"/dataset-versions/{dataset[\'dataset_version_id\']}/manifest").json()\n    assert manifest["counts"] == {"train": 1, "validation": 0, "test": 0}\n    frozen = client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/freeze")\n    assert frozen.json()["status"] == "frozen" and frozen.json()["manifest_checksum"]\n    blocked = add_item(client, dataset["dataset_version_id"], staging, original)\n    assert blocked.status_code == 409 and blocked.json()["code"] == "DATASET_VERSION_IMMUTABLE"\n\n\ndef test_only_approved_staging_can_enter_dataset(client):\n    sample_id = sample(client); original = upload(client, sample_id); dataset = version(client)\n    response = client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/items", json={\n        "staging_item_id": "missing", "image_asset_id": original["asset_id"], "split": "train"})\n    assert response.status_code == 404\n\n\ndef test_test_sample_can_never_move_to_training(client):\n    _, original, staging = approved_value_case(client); first = version(client, "dataset_test_v1")\n    assert add_item(client, first["dataset_version_id"], staging, original, "test").status_code == 201\n    second = version(client, "dataset_test_v2")\n    blocked = add_item(client, second["dataset_version_id"], staging, original, "train")\n    assert blocked.status_code == 409 and blocked.json()["code"] == "TEST_SET_IMMUTABLE"\n\n\ndef test_same_site_group_cannot_cross_splits(client):\n    _, first_image, first_stage = approved_value_case(client, area="shared-area")\n    _, second_image, second_stage = approved_value_case(client, area="shared-area")\n    dataset = version(client, "geo_safe_v1")\n    assert add_item(client, dataset["dataset_version_id"], first_stage, first_image, "train", "site-04").status_code == 201\n    blocked = add_item(client, dataset["dataset_version_id"], second_stage, second_image, "validation", "site-04")\n    assert blocked.status_code == 409 and blocked.json()["code"] == "GEOGRAPHIC_SPLIT_LEAKAGE"\n\n\ndef test_mask_dimensions_are_checked_before_dataset_add(client):\n    sample_id = sample(client); original = upload(client, sample_id, size=(32,32))\n    prediction = client.post(f"/samples/{sample_id}/predictions", json={"model_type":"coverage","model_version":"coverage_vx",\n        "predicted_value":{"coverage_percent":40},"confidence":.7,"is_placeholder":False}).json()\n    reviewed = upload(client, sample_id, kind="reviewed_mask", value=200, source=original["asset_id"], size=(16,16))\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={"field_name":"coverage_mask",\n        "proposed_asset_id":reviewed["asset_id"],"reason":"A manually traced mask shows the corrected visible boundary.","submitted_by":"field-user"}).json()\n    for reviewer in ("r-a","r-b"):\n        result=client.post(f"/reviews/{correction[\'correction_id\']}/decision",json={"reviewer_id":reviewer,"decision":"approved","reviewer_notes":"Independent boundary inspection confirms this mask."})\n    dataset=version(client,"dimension_check_v1"); blocked=add_item(client,dataset["dataset_version_id"],result.json()["staging_item"],original)\n    assert blocked.status_code == 422 and blocked.json()["code"] == "ANNOTATION_DIMENSION_MISMATCH"\n\n\ndef test_zip_export_contains_images_masks_metadata_geojson_and_manifest(client):\n    _, original, staging = approved_mask_case(client); dataset = version(client, "coverage_export_v1")\n    assert add_item(client, dataset["dataset_version_id"], staging, original).status_code == 201\n    client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/freeze")\n    exported = client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/export", json={"export_format":"zip"})\n    assert exported.status_code == 201, exported.text\n    content = client.get(f"/exports/{exported.json()[\'export_id\']}/content").content\n    assert sha256(content).hexdigest() == exported.json()["checksum"]\n    with ZipFile(BytesIO(content)) as archive:\n        names = set(archive.namelist())\n        assert {"manifest.json", "metadata.csv", "records.geojson"} <= names\n        assert any(name.startswith("images/") for name in names) and any(name.startswith("masks/") for name in names)\n\n\n@pytest.mark.parametrize("export_format, suffix", [("csv", "metadata.csv"), ("json", "manifest.json"), ("geojson", "records.geojson")])\ndef test_individual_export_formats(client, export_format, suffix):\n    _, original, staging = approved_value_case(client); dataset = version(client, f"format_{export_format}_v1")\n    add_item(client, dataset["dataset_version_id"], staging, original); client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/freeze")\n    record = client.post(f"/dataset-versions/{dataset[\'dataset_version_id\']}/export", json={"export_format":export_format}).json()\n    assert record["storage_path"].endswith(suffix)\n    assert client.get(f"/exports/{record[\'export_id\']}/content").status_code == 200\n\n\ndef test_audit_events_are_hash_chained_and_trace_core_actions(client):\n    sample_id, _, _ = approved_value_case(client)\n    events = client.get(f"/audit-events?sample_id={sample_id}").json()\n    event_types = {event["event_type"] for event in events}\n    assert {"SAMPLE_CREATED", "IMAGE_UPLOADED", "PREDICTION_CREATED", "CORRECTION_SUBMITTED", "CORRECTION_APPROVED"} <= event_types\n    assert client.get("/audit-events/verify").json()["valid"] is True\n\n\ndef test_standard_error_envelope_and_failure_catalog(client):\n    missing = client.get("/samples/not-real")\n    assert missing.status_code == 404\n    assert {"status", "code", "message", "severity", "retry_allowed", "details", "detail"} <= set(missing.json())\n    catalog = client.get("/failure-codes").json()\n    assert {"IMAGE_TOO_BLURRY", "MODEL_NOT_AVAILABLE", "LOCATION_MISSING", "DATABASE_WRITE_FAILED"} <= set(catalog)\n\n\ndef complete_card(name="Coverage Candidate"):\n    return {"name":name,"purpose":"Estimate visible lichen coverage","input":"RGB image","output":"Coverage percent and mask",\n        "training_data":"Versioned reviewed images","validation_data":"Held-out site groups","known_failure_cases":["Background texture"],\n        "intended_use":"Screening and field research","prohibited_use":"Regulatory enforcement without expert review"}\n\n\ndef test_model_registry_card_update_and_retirement_traceability(client):\n    created = client.post("/models", json={"model_type":"growth_form","version":"growthform_v2","status":"validation_candidate",\n        "validation_status":"partial","card":complete_card("Growth Form v2"),"metrics":{"macro_accuracy":.85},"threshold":.8})\n    assert created.status_code == 201, created.text\n    retired = client.patch(f"/models/{created.json()[\'model_id\']}", json={"status":"retired","validation_status":"complete"})\n    assert retired.json()["status"] == "retired"\n    assert client.get(f"/models/{created.json()[\'model_id\']}").json()["metrics"]["macro_accuracy"] == .85\n\n\ndef test_incomplete_model_card_is_rejected(client):\n    response = client.post("/models", json={"model_type":"growth_form","version":"bad-card","status":"development","card":{"name":"bad"}})\n    assert response.status_code == 422 and response.json()["code"] == "VALIDATION_ERROR"\n\n\ndef test_production_model_requires_frozen_dataset(client):\n    dataset = version(client, "training_draft_v1")\n    response = client.post("/models", json={"model_type":"coverage","version":"coverage_prod_v1","status":"production",\n        "dataset_version_id":dataset["dataset_version_id"],"validation_status":"complete","card":complete_card()})\n    assert response.status_code == 422 and response.json()["code"] == "DATASET_NOT_FROZEN"\n\n\ndef test_attached_predictions_are_auto_registered(client):\n    sample_id = sample(client)\n    response = client.post(f"/samples/{sample_id}/predictions", json={"model_type":"chlorosis","model_version":"chlorosis_candidate_v1",\n        "predicted_value":{"severity":"mild"},"confidence":.7,"is_placeholder":False})\n    assert response.status_code == 201\n    entries = client.get("/models?model_type=chlorosis").json()\n    assert any(entry["version"] == "chlorosis_candidate_v1" for entry in entries)\n', 'tests/test_features_11_16.py': 'from io import BytesIO\nfrom pathlib import Path\n\nimport pytest\nfrom fastapi.testclient import TestClient\nfrom PIL import Image\n\nfrom app.main import create_app\n\n\ndef checker(size=(96, 96)) -> bytes:\n    image = Image.new("L", size)\n    pixels = image.load()\n    for y in range(size[1]):\n        for x in range(size[0]): pixels[x, y] = 35 if (x + y) % 2 else 220\n    buffer = BytesIO(); image.save(buffer, format="PNG"); return buffer.getvalue()\n\n\nclass FakeAdapter:\n    def __init__(self, model_type, version, predicted, confidence=.8, placeholder=False):\n        self.model_type = model_type; self.model_version = version; self.predicted = predicted\n        self.confidence = confidence; self.is_placeholder = placeholder\n\n    def predict(self, image_path, context=None):\n        value = dict(self.predicted)\n        if self.model_type == "environmental_risk":\n            value["received_features"] = sorted((context or {}).keys())\n        return {"predicted_value": value, "confidence": self.confidence,\n            "model_version": self.model_version, "is_placeholder": self.is_placeholder, "warnings": []}\n\n\nclass FakeCoverage:\n    model_type = "coverage"; model_version = "coverage_v2"; is_placeholder = False\n    def predict(self, image_path, context=None):\n        with Image.open(image_path) as image: size = image.size\n        mask = Image.new("L", size, 255); buffer = BytesIO(); mask.save(buffer, format="PNG")\n        return {"coverage_percent": 39.0, "confidence": .9, "mask_bytes": buffer.getvalue(),\n            "model_version": self.model_version, "is_placeholder": False, "warnings": []}\n\n\n@pytest.fixture()\ndef client(tmp_path: Path):\n    adapters = {\n        "coverage": FakeCoverage(),\n        "growth_form": FakeAdapter("growth_form", "growthform_v1", {"class":"foliose", "class_probabilities":{"foliose":.84}}, .84),\n        "chlorosis": FakeAdapter("chlorosis", "chlorosis_candidate_v1", {"severity":"mild", "index":.27, "validation_status":"pending"}, .71),\n        "environmental_risk": FakeAdapter("environmental_risk", "risk_v3", {"class":"moderate", "score":61}, .8),\n    }\n    app = create_app(f"sqlite:///{tmp_path/\'features_11_16.db\'}", storage_root=tmp_path/"assets",\n        model_adapters=adapters, quality_thresholds={"min_width":8,"min_height":8,"min_blur_score":0,\n            "min_brightness":0,"max_brightness":1,"max_overexposed_fraction":1,"min_contrast":0})\n    with TestClient(app) as test_client: yield test_client\n\n\ndef make_sample(client, *, area="area-1", site="school", confidence=.55):\n    sample = client.post("/samples", json={"study_area_id":area,"user_id":"field-user","captured_at":"2026-07-31T12:00:00-04:00",\n        "latitude":35.7915,"longitude":-78.7811,"substrate_type":"tree_bark","site_type":site,"status":"submitted"}).json()\n    upload = client.post(f"/samples/{sample[\'sample_id\']}/assets", data={"asset_type":"original_image","created_by":"tester"},\n        files={"file":("original.png",checker(),"image/png")})\n    assert upload.status_code == 201, upload.text\n    prediction = client.post(f"/samples/{sample[\'sample_id\']}/predictions", json={"model_type":"growth_form","model_version":"growthform_external_v1",\n        "predicted_value":{"class":"fruticose","class_probabilities":{"fruticose":confidence}},"confidence":confidence,"is_placeholder":False}).json()\n    return sample, upload.json(), prediction\n\n\ndef pending_correction(client, *, area="area-1", confidence=.55):\n    sample, image, prediction = make_sample(client, area=area, confidence=confidence)\n    correction = client.post(f"/predictions/{prediction[\'prediction_id\']}/corrections", json={"field_name":"growth_form",\n        "proposed_value":"foliose","reason":"Visible leafy lobes support a foliose classification.","submitted_by":"field-user"})\n    assert correction.status_code == 201, correction.text\n    return sample, image, prediction, correction.json()\n\n\ndef test_active_learning_creates_three_ranked_queue_types_with_reasons(client):\n    make_sample(client, area="sparse-zone", confidence=.4)\n    response = client.post("/active-learning/recalculate", json={})\n    assert response.status_code == 200, response.text\n    rows = response.json(); assert {row["queue_type"] for row in rows} == {"annotation_review","geographic_sampling","class_balancing"}\n    assert all(row["priority_reasons"] and row["formula_version"] == "active_learning_priority_v0" for row in rows)\n    scores = [row["priority_score"] for row in rows]; assert scores == sorted(scores, reverse=True)\n\n\ndef test_active_learning_weights_are_versioned_and_configurable(client):\n    make_sample(client)\n    weights={"model_uncertainty":.4,"geographic_uncertainty":.2,"class_rarity":.2,"sample_quality":.1,"study_importance":.1}\n    rows=client.post("/active-learning/recalculate",json={"formula_version":"active_learning_priority_v0_test","weights":weights}).json()\n    assert all(row["formula_version"] == "active_learning_priority_v0_test" for row in rows)\n\n\ndef test_active_learning_item_status_workflow(client):\n    make_sample(client); rows=client.post("/active-learning/recalculate",json={}).json(); item=rows[0]\n    updated=client.patch(f"/active-learning/items/{item[\'item_id\']}",json={"status":"assigned"})\n    assert updated.json()["status"] == "assigned"\n    assert any(row["item_id"] == item["item_id"] for row in client.get("/active-learning/queue?status=assigned").json())\n\n\ndef test_reviewer_dashboard_filters_and_sorts_with_complete_context(client):\n    first=pending_correction(client,area="north",confidence=.45); pending_correction(client,area="south",confidence=.8)\n    client.post("/active-learning/recalculate",json={})\n    response=client.get("/reviewer-dashboard?model_type=growth_form&study_area=north&confidence_max=0.5&status=pending_review")\n    assert response.status_code == 200\n    rows=response.json()["items"]; assert len(rows)==1 and rows[0]["correction_id"]==first[3]["correction_id"]\n    assert rows[0]["original_image_url"].endswith("/content") and rows[0]["priority"] >= 0\n\n\ndef test_reviewer_can_adjust_value_without_overwriting_user_proposal(client):\n    _,_,_,correction=pending_correction(client)\n    decision=client.post(f"/reviews/{correction[\'correction_id\']}/decision",json={"reviewer_id":"reviewer-1","decision":"approved",\n        "reviewer_notes":"Leaf structure supports foliose, with reviewer normalization.","adjusted_value":"uncertain"})\n    assert decision.status_code == 200, decision.text\n    assert decision.json()["staging_item"]["verified_value"] == "uncertain"\n    original=client.get(f"/corrections/{correction[\'correction_id\']}").json()\n    assert original["proposed_value"] == "foliose"\n\n\ndef approved_coverage_ground_truth(client, sample_id, prediction_id, value=38.0):\n    correction=client.post(f"/predictions/{prediction_id}/corrections",json={"field_name":"coverage_percent","proposed_value":value,\n        "reason":"Manual grid count establishes reviewed ground truth coverage.","submitted_by":"field-user"}).json()\n    result=client.post(f"/reviews/{correction[\'correction_id\']}/decision",json={"reviewer_id":"reviewer-ground","decision":"approved",\n        "reviewer_notes":"Independent grid count confirms this value."})\n    assert result.status_code==200\n\n\ndef test_reanalysis_creates_new_prediction_and_comparison_uses_ground_truth(client):\n    sample,image,_=make_sample(client)\n    first=client.post(f"/samples/{sample[\'sample_id\']}/predictions",json={"model_type":"coverage","model_version":"coverage_v1",\n        "predicted_value":{"coverage_percent":44.0},"confidence":.7,"is_placeholder":False}).json()\n    approved_coverage_ground_truth(client,sample["sample_id"],first["prediction_id"],38.0)\n    second=client.post(f"/samples/{sample[\'sample_id\']}/reanalyze",json={"asset_id":image["asset_id"],"model_type":"coverage","model_version":"coverage_v2"})\n    assert second.status_code==201 and second.json()["prediction_id"] != first["prediction_id"]\n    comparison=client.get(f"/samples/{sample[\'sample_id\']}/model-comparison").json()["predictions"]\n    coverage=[row for row in comparison if row["model_type"]=="coverage"]\n    assert {row["model_version"] for row in coverage} == {"coverage_v1","coverage_v2"}\n    assert next(row for row in coverage if row["model_version"]=="coverage_v2")["absolute_error"] == 1.0\n\n\ndef test_model_to_model_comparison_identifies_improvement(client):\n    sample,image,_=make_sample(client)\n    first=client.post(f"/samples/{sample[\'sample_id\']}/predictions",json={"model_type":"coverage","model_version":"coverage_v1",\n        "predicted_value":{"coverage_percent":44.0},"confidence":.7,"is_placeholder":False}).json()\n    approved_coverage_ground_truth(client,sample["sample_id"],first["prediction_id"],38.0)\n    client.post(f"/samples/{sample[\'sample_id\']}/reanalyze",json={"asset_id":image["asset_id"],"model_type":"coverage","model_version":"coverage_v2"})\n    result=client.get("/models/compare?model_a=coverage_v1&model_b=coverage_v2").json()\n    assert result["shared_sample_count"]==1 and result["comparisons"][0]["result"]=="improved"\n\n\ndef test_unconnected_reanalysis_adapter_returns_standard_model_error(client):\n    sample,image,_=make_sample(client)\n    result=client.post(f"/samples/{sample[\'sample_id\']}/reanalyze",json={"asset_id":image["asset_id"],"model_type":"unknown","model_version":"x"})\n    assert result.status_code==503 and result.json()["code"]=="MODEL_NOT_AVAILABLE"\n\n\ndef test_quality_gate_stores_metrics_thresholds_and_decision(client):\n    sample,image,_=make_sample(client)\n    result=client.post(f"/assets/{image[\'asset_id\']}/quality")\n    assert result.status_code==201\n    body=result.json(); assert body["decision"]=="accepted" and {"blur_score","brightness","contrast"} <= set(body["metrics"])\n    assert body["quality_version"]=="image_quality_v0_configurable" and body["thresholds"]["min_width"]==8\n    assert client.get(f"/samples/{sample[\'sample_id\']}/quality").json()[0]["quality_assessment_id"]==body["quality_assessment_id"]\n\n\ndef test_quality_gate_rejects_obviously_too_small_image(tmp_path):\n    app=create_app(f"sqlite:///{tmp_path/\'quality.db\'}",storage_root=tmp_path/\'assets\')\n    with TestClient(app) as client:\n        sample=client.post("/samples",json={"captured_at":"2026-07-31T12:00:00-04:00"}).json()\n        tiny=client.post(f"/samples/{sample[\'sample_id\']}/assets",data={"asset_type":"original_image","created_by":"t"},\n            files={"file":("tiny.png",checker((8,8)),"image/png")}).json()\n        quality=client.post(f"/assets/{tiny[\'asset_id\']}/quality").json()\n        assert quality["decision"]=="rejected" and "IMAGE_TOO_SMALL" in quality["warnings"]\n\n\ndef test_unified_pipeline_runs_all_connected_models_and_records_steps(client):\n    sample,image,_=make_sample(client)\n    response=client.post("/analyze-sample",json={"sample_id":sample["sample_id"],"asset_id":image["asset_id"],\n        "run_models":["coverage","growth_form","chlorosis","environmental_risk"],\n        "geospatial_layer_versions":{"geomorphology":"geomorphology_2026_07","temporal_map":"temporal_v2"}})\n    assert response.status_code==200, response.text\n    body=response.json(); assert body["status"]=="completed" and set(body["results"]) >= {"image_quality","coverage","growth_form","chlorosis","environmental_risk"}\n    assert body["results"]["environmental_risk"]["received_features"]\n    run=client.get(f"/analysis-runs/{body[\'analysis_run_id\']}").json()\n    assert len(run["steps"])==5 and run["input_asset_checksum"]==image["checksum"]\n    assert run["geospatial_layer_versions"]["temporal_map"]=="temporal_v2"\n\n\ndef test_unified_pipeline_continues_after_optional_model_failure(client):\n    sample,image,_=make_sample(client)\n    body=client.post("/analyze-sample",json={"sample_id":sample["sample_id"],"asset_id":image["asset_id"],\n        "run_models":["growth_form","not_connected","environmental_risk"]}).json()\n    assert body["status"]=="completed_with_warnings" and body["results"]["growth_form"]["class"]=="foliose"\n    assert any(warning["code"]=="MODEL_NOT_AVAILABLE" for warning in body["warnings"])\n    run=client.get(f"/analysis-runs/{body[\'analysis_run_id\']}").json()\n    assert any(step["status"]=="failed_nonblocking" for step in run["steps"])\n\n\ndef test_unified_pipeline_stops_safely_when_quality_rejects(tmp_path):\n    app=create_app(f"sqlite:///{tmp_path/\'blocked.db\'}",storage_root=tmp_path/\'assets\')\n    with TestClient(app) as client:\n        sample=client.post("/samples",json={"captured_at":"2026-07-31T12:00:00-04:00"}).json()\n        image=client.post(f"/samples/{sample[\'sample_id\']}/assets",data={"asset_type":"original_image","created_by":"t"},\n            files={"file":("tiny.png",checker((8,8)),"image/png")}).json()\n        body=client.post("/analyze-sample",json={"sample_id":sample["sample_id"],"asset_id":image["asset_id"],"run_models":["coverage"]}).json()\n        assert body["status"]=="needs_retake" and "coverage" not in body["results"]\n\n\ndef test_confidence_is_decomposed_and_missing_values_are_explicit(client):\n    sample,_,_=make_sample(client)\n    response=client.post(f"/samples/{sample[\'sample_id\']}/confidence",json={"components":{"image_quality":.86,"coverage":None,\n        "growth_form":.84,"chlorosis":.71,"geospatial_completeness":.93,"risk_engine":.8,"feature_completeness":.8}})\n    assert response.status_code==201, response.text\n    body=response.json(); assert body["components"]["coverage"] is None and 0<body["overall_confidence"]<1\n    assert any("Coverage" in limitation for limitation in body["limitations"])\n\n\ndef test_invalid_confidence_component_is_rejected(client):\n    sample,_,_=make_sample(client)\n    response=client.post(f"/samples/{sample[\'sample_id\']}/confidence",json={"components":{"coverage":1.2}})\n    assert response.status_code==422 and response.json()["code"]=="INVALID_CONFIDENCE_COMPONENTS"\n', 'pytest.ini': '[pytest]\ntestpaths = tests\naddopts = -ra\n\n'}
for relative_path, content in PACKAGE_FILES.items():
    destination = PACKAGE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding='utf-8')
if str(PACKAGE_ROOT) not in sys.path: sys.path.insert(0, str(PACKAGE_ROOT))
print(f'Embedded backend ready: {len(PACKAGE_FILES)} files')


## 4. Create the persistent cumulative application

In [ ]:
for module_name in list(sys.modules):
    if module_name == 'app' or module_name.startswith('app.'): del sys.modules[module_name]
from app.main import create_app
persistent_app = create_app(DATABASE_URL, storage_root=ASSET_ROOT)
print('EcoSignal Features 1–16 application created.')
print('Default coverage adapter: clearly labeled development placeholder')


## 5. Run all cumulative verification

The 73-test suite rechecks Features 1–5 and validates every remaining feature: dataset split leakage safeguards, frozen manifests, four export formats, checksums, hash-chained provenance, standardized failure envelopes, model cards and status rules, active-learning queues, reviewer filters and adjustments, immutable reanalysis, model comparisons, configurable image quality, complete/partial pipeline execution, and decomposed confidence.


In [ ]:
import os, subprocess, sys
test_environment = os.environ.copy(); test_environment['PYTHONPATH'] = str(PACKAGE_ROOT)
completed = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PACKAGE_ROOT,
                           env=test_environment, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr: print(completed.stderr)
if completed.returncode != 0: raise RuntimeError('Cumulative feature verification failed.')
print('PASS: 73/73 cumulative Features 1–16 tests passed')


## 6. Run a safe end-to-end demonstration

This demo creates a real persisted sample and image, runs the configurable quality gate, executes the coverage adapter through `/analyze-sample`, stores the prediction/mask/run/steps/confidence, and prints the audit-chain result. The default coverage output is explicitly marked as a development placeholder and must not be treated as scientific evidence.


In [ ]:
from datetime import datetime
from io import BytesIO
from fastapi.testclient import TestClient
from PIL import Image

demo_image = Image.new('RGB', (128, 128))
pixels = demo_image.load()
for y in range(128):
    for x in range(128): pixels[x, y] = (35, 100, 45) if (x + y) % 2 else (190, 215, 175)
buffer = BytesIO(); demo_image.save(buffer, format='PNG')

with TestClient(persistent_app) as client:
    sample = client.post('/samples', json={
        'study_area_id': 'colab-demo-area', 'user_id': 'colab-demo-user',
        'captured_at': datetime.now().astimezone().isoformat(),
        'latitude': 35.7915, 'longitude': -78.7811,
        'substrate_type': 'tree_bark', 'site_type': 'roadside',
        'notes': 'Features 1–16 cumulative demonstration', 'status': 'submitted'
    }).json()
    image = client.post(f"/samples/{sample['sample_id']}/assets",
        data={'asset_type': 'original_image', 'created_by': 'colab-demo-user'},
        files={'file': ('demo_lichen.png', buffer.getvalue(), 'image/png')}).json()
    analysis = client.post('/analyze-sample', json={
        'sample_id': sample['sample_id'], 'asset_id': image['asset_id'],
        'run_models': ['coverage'], 'pipeline_version': 'ecosignal_pipeline_v1',
        'geospatial_layer_versions': {}
    }).json()
    run_detail = client.get(f"/analysis-runs/{analysis['analysis_run_id']}").json()
    audit_check = client.get('/audit-events/verify').json()
print('Sample:', sample['sample_id'])
print('Analysis:', analysis)
print('Recorded steps:', run_detail['steps'])
print('Audit chain:', audit_check)
assert analysis['results']['coverage']['is_placeholder'] is True
assert audit_check['valid'] is True


## 7. Connect your existing real models

The persistent app above intentionally starts with only the coverage development placeholder. To connect your real models without changing any endpoint or database schema, create adapters that expose `model_type`, `model_version`, `is_placeholder`, and `predict(image_path, context=None)`, then rebuild the app with `model_adapters={...}`. Each adapter should return `predicted_value`, `confidence`, `model_version`, `is_placeholder`, and optional `warnings`. Coverage adapters may instead return `coverage_percent` and `mask_bytes`, matching your existing U-Net boundary.

Do not set `is_placeholder=False` until the corresponding real artifact and validation status are accurate.


In [ ]:
# Template only — replace each body with your existing model loader and inference code.
class EcoSignalModelAdapter:
    model_type = 'growth_form'
    model_version = 'growthform_v1'
    is_placeholder = False

    def predict(self, image_path: str, context=None) -> dict:
        raise NotImplementedError('Connect your saved model here before registering this adapter.')

# Example after implementing real adapters:
# connected_app = create_app(
#     DATABASE_URL,
#     storage_root=ASSET_ROOT,
#     model_adapters={
#         'coverage': real_coverage_adapter,
#         'growth_form': real_growth_form_adapter,
#         'chlorosis': real_chlorosis_adapter,
#         'environmental_risk': real_risk_adapter,
#     },
# )


## 8. Feature and endpoint map

- **Features 1–5:** samples, predictions, assets, inference boundary, corrections, reviews, dataset staging
- **Feature 6:** `/dataset-versions`, item assignment, freeze, manifest
- **Feature 7:** dataset export as ZIP, CSV, JSON, or GeoJSON; `/exports/{id}`
- **Feature 8:** `/audit-events`, chain verification, `/analysis-runs/{id}`
- **Feature 9:** standardized error envelopes and `/failure-codes`
- **Feature 10:** `/models` registry, model cards, lifecycle status
- **Feature 11:** `/active-learning/recalculate`, ranked queues, status workflow
- **Feature 12:** `/reviewer-dashboard` with type/model/confidence/area/date/status filters
- **Feature 13:** `/samples/{id}/reanalyze`, sample/model comparisons
- **Feature 14:** `/assets/{id}/quality`, configurable stored thresholds
- **Feature 15:** `/analyze-sample`, per-step provenance, safe partial failures
- **Feature 16:** `/samples/{id}/confidence`, component values, weights, limitations


## 9. Optional: launch Swagger documentation

In [ ]:
import threading, time, uvicorn
from google.colab import output
threading.Thread(target=uvicorn.run, args=(persistent_app,),
    kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'info'}, daemon=True).start()
time.sleep(2)
docs_base_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
print('FastAPI docs:', f'{docs_base_url}/docs')
